# HalluciGuard v5: Retrieval-Augmented Hallucination Verification for Australian Visa QA

**Project type:** NLP research workflow and empirical feasibility study  
**Domain:** Australian visa information  
**Primary contribution:** verification of generated answers against trusted retrieved context  
**Code repository:** https://github.com/Devarsh-04/halluci-guard-nlp

This notebook is organised as a research narrative rather than a collection of isolated experiments. It follows the complete pipeline from data construction to retrieval, generation, semantic verification, baselines, robustness analysis, and final interpretation.

**Marker navigation guide**

| Section | Purpose |
|---|---|
| 1-3 | Define the problem and system architecture |
| 4 | Build and validate the knowledge base and labelled hallucination data |
| 5-7 | Implement retrieval, generation, and hallucination verification |
| 8-10 | Compare baselines and evaluate the full system |
| 11-15 | Analyse robustness, limitations, and future work |
| 16 | Appendix demonstrations and additional visual summaries |


## 2. Motivation & Problem Definition

Large language models can produce answers that are fluent but unsupported by the available evidence. In high-stakes information settings such as migration or visa guidance, this creates a practical risk: a user may trust an answer that changes a date, fee, eligibility condition, or legal requirement.

HalluciGuard studies this problem in a controlled domain-specific QA setting. The system retrieves relevant Australian visa passages, generates an answer with a lightweight model, and then verifies whether the answer is grounded in the retrieved evidence.

The project is deliberately framed as a feasibility study. The aim is not to build a production immigration assistant or a state-of-the-art generator. The central research question is:

> Can a lightweight multi-signal NLP verification pipeline detect unsupported or contradictory claims in generated visa answers more reliably than shallow text-matching baselines?

This framing keeps the contribution focused on semantic verification, hard-negative robustness, and careful evaluation.


## 3. System Overview

The pipeline contains five stages:

1. **Knowledge base construction:** curated domain passages covering Australian visa categories.
2. **Query analysis and retrieval:** POS/NER-assisted query processing, Sentence-BERT embeddings, FAISS search, and cross-encoder re-ranking.
3. **Answer generation:** Flan-T5-base produces concise answers from retrieved context.
4. **Hallucination verification:** NER/number checking, NLI entailment scoring, and embedding similarity are fused into a groundedness score.
5. **Evaluation:** shallow baselines, ablation, calibration, statistical testing, robustness probes, and error taxonomy.

The verification layer is the main technical contribution. Generation is included to create an end-to-end QA setting, but the report should not overstate generation quality.


### 3.1 Reproducible Environment

The notebook is designed for Google Colab with GPU enabled. The first cell installs the required NLP libraries and spaCy models. Runtime downloads are expected because the project uses transformer models, sentence encoders, FAISS, and spaCy pipelines.


In [ ]:
# ============================================================
# 1.1 — Install packages
# ============================================================
!pip install -q transformers sentence-transformers faiss-cpu spacy \
    datasets torch scikit-learn gensim nltk
!python -m spacy download en_core_web_sm -q
!python -m spacy download en_core_web_md -q

In [ ]:
# ============================================================
# 1.2 — Imports and configuration
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json, re, time, warnings, random
from itertools import product
from collections import Counter

warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    T5ForConditionalGeneration, T5Tokenizer,
)
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
import spacy
import nltk
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score
)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Device ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# --- spaCy ---
nlp_sm = spacy.load("en_core_web_sm")
nlp_md = spacy.load("en_core_web_md")

print("All dependencies loaded.")

## 4. Dataset Construction & Integrity Validation

This section builds two related data resources:

- A domain knowledge base used for retrieval and answer grounding.
- Labelled `(source, answer)` examples used to evaluate hallucination detection.

The labelled data includes both grounded paraphrases and hallucinated answers. The v5 expansion adds hard negatives and difficulty labels so the evaluation tests subtle numerical, entity, and unsupported-inference errors rather than only obvious contradictions.


### 4.1 Domain Knowledge Base

The knowledge base is intentionally domain-specific. This makes retrieval and verification interpretable: every generated answer should be traceable to a small set of trusted visa passages.


In [ ]:
# ============================================================
# 2.1 — Australian Visa FAQ Knowledge Base (120 passages)
# ============================================================
knowledge_base = [
    # STUDENT VISA (20)
    {"id":"sv001","category":"Student Visa","text":"The Student Visa Subclass 500 allows international students to study full-time in Australia at a recognised education institution. The visa is valid for up to 5 years depending on the course duration. Applicants must be at least 6 years old to apply."},
    {"id":"sv002","category":"Student Visa","text":"To apply for a Student Visa 500, applicants must provide a Confirmation of Enrolment (CoE) from their education institution. The institution must be registered on CRICOS (Commonwealth Register of Institutions and Courses for Overseas Students). If applying for multiple courses as a package, CoEs for all courses must be submitted."},
    {"id":"sv003","category":"Student Visa","text":"Student visa applicants must satisfy the Genuine Student (GS) requirement, which replaced the previous Genuine Temporary Entrant (GTE) requirement. Applicants must answer targeted questions demonstrating their genuine intention to study in Australia temporarily and comply with visa conditions."},
    {"id":"sv004","category":"Student Visa","text":"Student visa holders can work up to 48 hours per fortnight while their course is in session. Work rights commence once the course has started, not when the visa is granted. During scheduled course breaks, students may work unlimited hours."},
    {"id":"sv005","category":"Student Visa","text":"All student visa holders must maintain Overseas Student Health Cover (OSHC) for the entire duration of their stay in Australia. Maintaining OSHC is a mandatory visa condition, and failure to do so may result in visa cancellation."},
    {"id":"sv006","category":"Student Visa","text":"The base application fee for the Student Visa Subclass 500 is AUD 1,600. Additional charges apply for family members included in the application. Applicants must also demonstrate sufficient funds for living expenses, currently set at AUD 29,710 per year."},
    {"id":"sv007","category":"Student Visa","text":"Student visa holders must notify their education provider of their residential address within 7 days of arriving in Australia. They must maintain satisfactory attendance and course progression, and remain enrolled in a CRICOS-registered course throughout their stay."},
    {"id":"sv008","category":"Student Visa","text":"After completing studies in Australia, students may be eligible for a Temporary Graduate Visa (Subclass 485), which allows them to work in Australia temporarily to gain practical experience in their field of study."},
    {"id":"sv009","category":"Student Visa","text":"Student visa applicants must provide proof of English language proficiency through approved tests such as IELTS, TOEFL iBT, PTE Academic, or Cambridge English. The Department of Home Affairs does not accept results from tests taken at home or online, including TOEFL iBT Home Edition and IELTS Online."},
    {"id":"sv010","category":"Student Visa","text":"For 2026, Australia has implemented a National Planning Level (NPL) cap of 295,000 international student places. Applications from certain countries including India, Nepal, and Bhutan may face higher levels of scrutiny under Evidence Level 3, requiring manual verification of bank statements and transcripts."},
    {"id":"sv011","category":"Student Visa","text":"Students can apply to study two or more courses on a single Student Visa 500 where there is clear progression from one course to another. This is known as course packaging. Course gaps must be less than two calendar months between packaged courses."},
    {"id":"sv012","category":"Student Visa","text":"Student visa applicants under 18 years of age must have welfare arrangements approved by the Department. A parent, legal guardian, or approved relative must be nominated. Alternatively, the education provider may issue a CAAW (Confirmation of Appropriate Accommodation and Welfare) letter."},
    {"id":"sv013","category":"Student Visa","text":"Secondary school students applying for a Student Visa 500 must meet age-for-year requirements: younger than 17 for Year 9, younger than 18 for Year 10, younger than 19 for Year 11, and younger than 20 for Year 12."},
    {"id":"sv014","category":"Student Visa","text":"Student visa holders who wish to change their education provider must ensure their new course is also registered on CRICOS. Changing courses without maintaining visa conditions may result in a breach that could lead to visa cancellation."},
    {"id":"sv015","category":"Student Visa","text":"Dependants of student visa holders, such as spouses and children, may have different work rights depending on the principal applicant's course level. Dependants of students in higher education doctoral or master's programs generally receive unlimited work rights."},
    {"id":"sv016","category":"Student Visa","text":"Student visa processing times vary depending on the applicant's country of origin and the assessment level. Average processing times range from a few weeks to several months. Applicants can track their application status through ImmiAccount."},
    {"id":"sv017","category":"Student Visa","text":"A student visa may be cancelled if the holder fails to comply with visa conditions, including failing to maintain enrolment, not maintaining OSHC, working more than the permitted hours, or providing false information in their application."},
    {"id":"sv018","category":"Student Visa","text":"The Genuine Student requirement asks applicants to address questions about their understanding of their chosen course, their reasons for studying in Australia, and how the course relates to their future career plans. Generic or AI-generated responses may result in refusal."},
    {"id":"sv019","category":"Student Visa","text":"Financial evidence for a student visa must demonstrate access to funds for tuition fees, living costs (AUD 29,710 per year), school-aged dependants' costs (AUD 8,296 per year), and return airfares. Funds must be genuinely available and may be verified by the Department."},
    {"id":"sv020","category":"Student Visa","text":"The Student Guardian Visa (Subclass 590) allows a parent, legal guardian, or approved relative to stay in Australia as a guardian of a student visa holder who is under 18 years of age. The guardian must maintain adequate health insurance and cannot work more than 48 hours per fortnight."},
    # VISITOR VISA (12)
    {"id":"vv001","category":"Visitor Visa","text":"The Visitor Visa Subclass 600 is for people who want to visit Australia temporarily for tourism, to visit family and friends, or for business visitor activities. The visa can be granted for stays of 3, 6, or 12 months depending on the applicant's circumstances."},
    {"id":"vv002","category":"Visitor Visa","text":"Visitor visa holders are generally not permitted to work in Australia. The visa is strictly for visiting purposes including tourism, visiting family and friends, attending business meetings or conferences, or participating in non-work activities."},
    {"id":"vv003","category":"Visitor Visa","text":"To apply for a Visitor Visa 600, applicants must demonstrate they have genuine reasons to visit Australia, have sufficient funds for their stay, have adequate health insurance, and intend to leave Australia before their visa expires."},
    {"id":"vv004","category":"Visitor Visa","text":"The Electronic Travel Authority (ETA) Subclass 601 is available for passport holders from certain eligible countries. It allows for short visits to Australia for tourism or business purposes for up to 3 months at a time within a 12-month validity period."},
    {"id":"vv005","category":"Visitor Visa","text":"The eVisitor Visa Subclass 651 is available to passport holders from European Union member states and several other European countries. It allows visits of up to 3 months at a time for tourism or business within a 12-month validity period, and there is no application charge."},
    {"id":"vv006","category":"Visitor Visa","text":"Business visitors to Australia on a Visitor Visa 600 can attend meetings, conferences, and trade fairs, and conduct general business enquiries. They cannot perform any work that would normally be done by an Australian worker or receive payment from an Australian source."},
    {"id":"vv007","category":"Visitor Visa","text":"The Visitor Visa 600 has a Tourist stream, a Business Visitor stream, a Sponsored Family stream, and an Approved Destination Status stream. Each stream has different eligibility criteria and evidence requirements."},
    {"id":"vv008","category":"Visitor Visa","text":"Visitor visa applicants may need to provide evidence of ties to their home country, such as employment, property ownership, or family connections, to demonstrate their intention to return home after their visit."},
    {"id":"vv009","category":"Visitor Visa","text":"Visitor visa holders cannot study for more than 3 months on their visa. If they wish to study for longer, they must apply for a Student Visa Subclass 500 instead."},
    {"id":"vv010","category":"Visitor Visa","text":"The application charge for a Visitor Visa 600 Tourist stream applied from outside Australia starts from AUD 190. Fees vary depending on the stream and where the application is lodged."},
    {"id":"vv011","category":"Visitor Visa","text":"Visitor visa holders must have adequate health insurance for their stay in Australia. While not always a mandatory visa condition, the Department strongly recommends it as medical treatment in Australia can be very expensive for non-residents."},
    {"id":"vv012","category":"Visitor Visa","text":"Sponsored Family stream applicants need an Australian citizen or permanent resident sponsor who provides a bond. The bond amount varies and is refundable if the visa holder complies with all conditions and departs Australia before the visa expires."},
    # PARTNER VISA (10)
    {"id":"pv001","category":"Partner Visa","text":"The Partner Visa allows the spouse or de facto partner of an Australian citizen, permanent resident, or eligible New Zealand citizen to live in Australia. There are onshore subclasses (820/801) and offshore subclasses (309/100)."},
    {"id":"pv002","category":"Partner Visa","text":"Partner visa applicants must provide evidence of a genuine and continuing relationship with their sponsoring partner. This includes evidence of shared finances, living arrangements, social recognition as a couple, and mutual commitment to a shared life."},
    {"id":"pv003","category":"Partner Visa","text":"The partner visa application is a two-stage process. First, a temporary partner visa is granted, and after approximately two years, the applicant is assessed for a permanent partner visa, provided the relationship is still genuine and continuing."},
    {"id":"pv004","category":"Partner Visa","text":"The combined application charge for the Partner Visa is AUD 9,095, making it one of the most expensive visa categories. This fee covers both the temporary and permanent stages of the application."},
    {"id":"pv005","category":"Partner Visa","text":"Partner visa sponsors must be approved by the Department. Sponsors may be refused if they have relevant criminal convictions, have previously sponsored a partner within the last 5 years, or have sponsored two or more partners in total."},
    {"id":"pv006","category":"Partner Visa","text":"De facto relationships must have existed for at least 12 months before the visa application is lodged, unless there are compelling or compassionate circumstances or the relationship is registered with an Australian state or territory."},
    {"id":"pv007","category":"Partner Visa","text":"Prospective Marriage Visa (Subclass 300) allows the fiance of an Australian citizen, permanent resident, or eligible NZ citizen to come to Australia to marry and then apply for a Partner Visa onshore. The couple must marry within 9 months of the visa being granted."},
    {"id":"pv008","category":"Partner Visa","text":"Partner visa applicants with children from previous relationships can include dependent children in their application. Step-children and adopted children may also be included, subject to meeting eligibility requirements."},
    {"id":"pv009","category":"Partner Visa","text":"Partner visa holders receive work and study rights in Australia while on the temporary partner visa. They also receive access to Medicare, Australia's public health system, from the date of visa grant."},
    {"id":"pv010","category":"Partner Visa","text":"Processing times for partner visas are currently 8 to 28 months for the temporary stage and 14 to 24 months for the permanent stage. Priority processing is not available for partner visa applications."},
    # WORK VISA (12)
    {"id":"wv001","category":"Work Visa","text":"The Skilled Independent Visa (Subclass 189) is a points-tested permanent visa for skilled workers who are not sponsored by an employer, state, or family member. Applicants must be invited to apply through the SkillSelect system and meet the minimum points score of 65."},
    {"id":"wv002","category":"Work Visa","text":"The Temporary Skill Shortage Visa (Subclass 482) allows employers to sponsor skilled overseas workers when they cannot find suitably qualified Australian workers. The visa has a Short-term stream (up to 2 years) and a Medium-term stream (up to 4 years)."},
    {"id":"wv003","category":"Work Visa","text":"The Working Holiday Visa (Subclass 417) allows young adults aged 18 to 30 (or 35 for some countries) from eligible countries to holiday and work in Australia for up to 12 months. Visa holders can work for the same employer for up to 6 months."},
    {"id":"wv004","category":"Work Visa","text":"The Employer Nomination Scheme Visa (Subclass 186) is a permanent visa for skilled workers nominated by their employer. Applicants must be nominated for a position on the relevant skilled occupation list and meet skill and English language requirements."},
    {"id":"wv005","category":"Work Visa","text":"The Skilled Nominated Visa (Subclass 190) is a points-tested permanent visa for skilled workers nominated by an Australian state or territory government. Nomination adds 5 points to the applicant's points score, and applicants must commit to living in the nominating state for at least 2 years."},
    {"id":"wv006","category":"Work Visa","text":"The Skilled Work Regional Visa (Subclass 491) is a points-tested provisional visa for skilled workers nominated by a state or territory government or sponsored by an eligible family member living in regional Australia. The visa is valid for 5 years."},
    {"id":"wv007","category":"Work Visa","text":"Points-tested skilled visas award points for factors including age (25-32 years receives maximum points), English proficiency, work experience, qualifications, and partner skills. The minimum points score required is 65."},
    {"id":"wv008","category":"Work Visa","text":"Skills assessment is required for most skilled visa applications. The assessing authority depends on the nominated occupation. For example, Engineers Australia assesses engineering occupations, and the Australian Computer Society (ACS) assesses IT occupations."},
    {"id":"wv009","category":"Work Visa","text":"The Global Talent Visa (Subclass 858) is for highly skilled professionals who can demonstrate international recognition in their field. Target sectors include technology, health, resources, agri-food, education, and financial services."},
    {"id":"wv010","category":"Work Visa","text":"Working Holiday Visa holders who perform specified work in regional Australia for at least 3 months during their first year may be eligible for a second-year visa. Specified work includes agriculture, mining, construction, and certain other industries."},
    {"id":"wv011","category":"Work Visa","text":"The Training Visa (Subclass 407) allows people to participate in occupational training or professional development in Australia. The training must be workplace-based and either improve skills for the applicant's current occupation or meet a registration or licensing requirement."},
    {"id":"wv012","category":"Work Visa","text":"Temporary Graduate Visa (Subclass 485) has two streams: the Graduate Work stream requires an occupation on the skilled occupation list, while the Post-Study Work stream is based on the qualification obtained. Post-Study Work stream validity ranges from 2 to 6 years depending on the qualification."},
    # CITIZENSHIP (10)
    {"id":"cz001","category":"Citizenship","text":"To be eligible for Australian citizenship by conferral, applicants generally must be permanent residents, have lived in Australia for at least four years including at least 12 months as a permanent resident, and be of good character."},
    {"id":"cz002","category":"Citizenship","text":"Australian citizenship applicants must pass a citizenship test that assesses their knowledge of Australia, including its values, history, traditions, and national symbols. The test is conducted in English and consists of 20 multiple-choice questions with a pass mark of 75 percent."},
    {"id":"cz003","category":"Citizenship","text":"After a citizenship application is approved, applicants must attend a citizenship ceremony where they make the Australian Citizenship Pledge. Invitations to the ceremony are usually sent about 4 weeks before the event. Wait times vary by location."},
    {"id":"cz004","category":"Citizenship","text":"Citizenship applications can be tracked through ImmiAccount. Current processing times for citizenship applications vary and are published on the Department of Home Affairs website. Processing times do not include the time from approval to ceremony."},
    {"id":"cz005","category":"Citizenship","text":"Children born in Australia to at least one parent who is an Australian citizen or permanent resident at the time of the child's birth automatically acquire Australian citizenship. Children born in Australia to non-citizen parents may acquire citizenship on their 10th birthday if they have lived most of their life in Australia."},
    {"id":"cz006","category":"Citizenship","text":"Australian citizenship can be acquired by descent if at least one parent was an Australian citizen at the time of the child's birth. Applications for citizenship by descent must be lodged before the person turns 25 years of age."},
    {"id":"cz007","category":"Citizenship","text":"The general residence requirement for citizenship is four years of lawful residence in Australia, with no more than 12 months spent outside Australia during that period, including no more than 90 days outside Australia in the 12 months before applying."},
    {"id":"cz008","category":"Citizenship","text":"Applicants for citizenship must demonstrate basic competence in the English language. The citizenship test itself serves as the English language assessment for most applicants. Special provisions exist for applicants over 60 years of age or with hearing, speech, or sight disabilities."},
    {"id":"cz009","category":"Citizenship","text":"The application fee for Australian citizenship by conferral is AUD 490. This fee is non-refundable if the application is refused. There is no additional fee for children included in a parent's citizenship application."},
    {"id":"cz010","category":"Citizenship","text":"Dual citizenship is permitted in Australia. Applicants for Australian citizenship are not required to renounce their existing citizenship. However, they should check whether their country of origin permits dual citizenship."},
    # APPLICATION PROCESS (12)
    {"id":"ap001","category":"Application Process","text":"Most visa applications must be submitted online through ImmiAccount. ImmiAccount is the Department of Home Affairs' online portal for visa and citizenship applications. Some applications may be submitted on paper, but online submission is strongly encouraged."},
    {"id":"ap002","category":"Application Process","text":"All documents not in English must be translated by accredited translators. In Australia, translators must be accredited by the National Accreditation Authority for Translators and Interpreters (NAATI). Both the original and translated documents must be submitted."},
    {"id":"ap003","category":"Application Process","text":"Visa processing times vary depending on the visa type, individual circumstances, and caseload. Applications may take longer if they contain incorrect information, missing documents, or require further assessment. The Department cannot expedite applications except in limited circumstances."},
    {"id":"ap004","category":"Application Process","text":"Some visa applicants may be required to provide biometrics including fingerprints and facial photographs. Biometric collection services are provided by VFS Global through Australian Biometrics Collection Centres (ABCCs) in many countries worldwide."},
    {"id":"ap005","category":"Application Process","text":"Only registered migration agents and certain legal practitioners are authorised to provide immigration assistance for a fee. Using an unregistered agent is illegal. The Office of the Migration Agents Registration Authority (OMARA) maintains a register of agents."},
    {"id":"ap006","category":"Application Process","text":"Visa Application Charge (VAC) refunds are available only in certain limited circumstances, such as duplicate payments or system errors. There is no standard timeframe for processing refund requests. Voluntary withdrawal of an application does not guarantee a refund."},
    {"id":"ap007","category":"Application Process","text":"If a visa application is refused, the applicant may have the right to seek review at the Administrative Review Tribunal (ART), formerly known as the AAT. The refusal notification will detail whether review rights exist and the timeframe for lodging a review."},
    {"id":"ap008","category":"Application Process","text":"Applicants should submit a complete and decision-ready application to avoid delays. The 'Check twice, submit once' initiative encourages applicants to review their application thoroughly before lodging. Incomplete applications may be refused without further assessment."},
    {"id":"ap009","category":"Application Process","text":"Visa applicants can withdraw their application at any time through ImmiAccount. Withdrawal is voluntary and permanent. A new application must be lodged with a new Visa Application Charge."},
    {"id":"ap010","category":"Application Process","text":"The Department may request additional information or documents during the processing of an application. Applicants are given a deadline to respond. Failure to respond may result in the application being decided on the information available, which could lead to refusal."},
    {"id":"ap011","category":"Application Process","text":"Ministerial Direction 115 sets out the order of priority for processing student visa applications. Applications from lower-risk countries and higher-quality education providers are generally processed more quickly."},
    {"id":"ap012","category":"Application Process","text":"Visa label-free travel means that visa information is stored electronically. Most Australian visas are now granted without a physical visa label in the passport. Travellers should carry their visa grant notification letter when travelling."},
    # HEALTH AND CHARACTER (10)
    {"id":"hc001","category":"Health and Character","text":"Visa applicants may need to undergo health examinations conducted by a Bupa Medical Visa Services panel physician. Health examination results are usually valid for 12 months. If applicants are asked to sign a health undertaking, the results are valid for 6 months."},
    {"id":"hc002","category":"Health and Character","text":"Applicants must meet character requirements for most Australian visas. This typically includes providing police clearance certificates from every country where the applicant has lived for 12 months or more in the last 10 years."},
    {"id":"hc003","category":"Health and Character","text":"Australian police checks can be obtained through the Australian Federal Police (AFP) National Police Check application. State or territory police certificates are not accepted by the Department of Home Affairs for visa purposes."},
    {"id":"hc004","category":"Health and Character","text":"Visa holders must notify the Department of Home Affairs about any changes in their circumstances, including new passports, changes of address, changes to contact details, and changes in relationship status. Changes can be notified through the Update Us tab in ImmiAccount."},
    {"id":"hc005","category":"Health and Character","text":"The health requirement ensures that visa applicants do not have a condition that would be a risk to Australian public health, prejudice access to healthcare for Australian citizens, or cost the Australian community in healthcare or community services."},
    {"id":"hc006","category":"Health and Character","text":"Health waivers may be granted in certain circumstances if the applicant's condition does not meet the health requirement but the expected costs are below a certain threshold or there are compelling reasons to waive the requirement."},
    {"id":"hc007","category":"Health and Character","text":"A person may fail the character requirement if they have a substantial criminal record, have been convicted of a sexually-based offence involving a child, have been involved in people smuggling or human trafficking, or are assessed as a risk to the Australian community."},
    {"id":"hc008","category":"Health and Character","text":"Chest X-rays for tuberculosis screening are required for applicants over 11 years of age from countries with high TB rates. Additional health examinations may include blood tests, HIV testing, and hepatitis screening depending on the applicant's circumstances."},
    {"id":"hc009","category":"Health and Character","text":"The Department uses the Health Assessment Record (HAP ID) system to manage health examinations. Applicants generate a HAP ID through ImmiAccount or eMedical and present it to the panel physician at their health examination appointment."},
    {"id":"hc010","category":"Health and Character","text":"Character assessments may also involve security checks conducted by ASIO (Australian Security Intelligence Organisation). These checks are not visible to applicants but may cause delays in visa processing, particularly for applicants from certain countries."},
    # VISA CONDITIONS (8)
    {"id":"vc001","category":"Visa Conditions","text":"Visa Entitlement Verification Online (VEVO) allows visa holders, employers, education providers, and other organisations to check visa details and conditions. VEVO shows details relating to the current in-effect visa only, not previous or expired visas."},
    {"id":"vc002","category":"Visa Conditions","text":"Breaching visa conditions may result in visa cancellation. Common breaches include working more hours than permitted, not maintaining required health insurance, failing to maintain course enrolment for students, or providing false or misleading information."},
    {"id":"vc003","category":"Visa Conditions","text":"If you are in Australia without a valid visa, you become an unlawful non-citizen and could face serious consequences including immigration detention and removal from Australia. The Status Resolution Support Service can help people in this situation."},
    {"id":"vc004","category":"Visa Conditions","text":"Condition 8105 limits the holder to working no more than 48 hours per fortnight when the holder's course of study or training is in session. This is the standard work condition applied to most student visa holders."},
    {"id":"vc005","category":"Visa Conditions","text":"Condition 8501 requires the visa holder to maintain adequate health insurance during their stay in Australia. For student visa holders, this means maintaining OSHC. For other visa holders, it may mean private health insurance."},
    {"id":"vc006","category":"Visa Conditions","text":"Condition 8202 requires student visa holders to remain enrolled in a registered course, maintain satisfactory attendance, and maintain satisfactory academic results as determined by the education provider."},
    {"id":"vc007","category":"Visa Conditions","text":"No-further-stay condition 8503 prevents the visa holder from applying for most substantive visas while in Australia. If this condition is applied, the holder must leave Australia before the visa expires and apply for any new visa from outside Australia."},
    {"id":"vc008","category":"Visa Conditions","text":"Condition 8516 requires the visa holder to continue to satisfy the criteria under which the visa was originally granted. For student visa holders, this includes maintaining adequate financial capacity and OSHC coverage."},
    # FAMILY AND MINORS (8)
    {"id":"fm001","category":"Family and Minors","text":"Visa applicants under 18 years old must provide Form 1229 (Consent to grant an Australian visa to a child under 18 years). Both parents must complete this form and provide identification documents such as passport copies."},
    {"id":"fm002","category":"Family and Minors","text":"If a child under 18 is travelling to Australia with someone other than a parent, the guardian must complete Form 1257 (Undertaking Declaration) and provide their identification documents. This must be submitted with the visa application."},
    {"id":"fm003","category":"Family and Minors","text":"The Parent Visa (Subclass 103) allows parents of settled Australian citizens, permanent residents, or eligible NZ citizens to live in Australia permanently. Processing times for this visa are extremely long, often exceeding 30 years due to capping."},
    {"id":"fm004","category":"Family and Minors","text":"The Contributory Parent Visa (Subclass 143) is a faster alternative to the standard Parent Visa but has a significantly higher cost, with a second instalment of approximately AUD 47,755. Processing times are typically 5 to 7 years."},
    {"id":"fm005","category":"Family and Minors","text":"The Child Visa (Subclass 101/802) allows dependent children of Australian citizens, permanent residents, or eligible NZ citizens to live permanently in Australia. The child must be dependent on the parent and under 18, or between 18 and 25 and a full-time student."},
    {"id":"fm006","category":"Family and Minors","text":"The Orphan Relative Visa (Subclass 117/837) is for children under 18 whose parents are unable to care for them due to death, permanent incapacity, or unknown whereabouts. The sponsoring relative must be an Australian citizen or permanent resident."},
    {"id":"fm007","category":"Family and Minors","text":"The Carer Visa (Subclass 836) allows individuals to remain in Australia to provide care for an Australian citizen, permanent resident, or eligible NZ citizen who has a medical condition requiring long-term personal care."},
    {"id":"fm008","category":"Family and Minors","text":"The Aged Dependent Relative Visa (Subclass 114/838) is for single applicants who are old enough to receive the Australian Age Pension and who are dependent on their relative in Australia for financial support."},
    # BRIDGING VISAS (8)
    {"id":"bv001","category":"Bridging Visa","text":"Bridging visas are temporary visas that allow people to stay in Australia lawfully while their substantive visa application is being processed. They generally come with specific conditions about work rights and travel restrictions."},
    {"id":"bv002","category":"Bridging Visa","text":"Bridging Visa A (BVA) is automatically granted to visa applicants who hold a valid substantive visa when they apply for a new visa in Australia. It allows them to remain lawfully in Australia while their new application is processed."},
    {"id":"bv003","category":"Bridging Visa","text":"Bridging Visa B (BVB) allows the holder to travel outside Australia and return during its validity period while their substantive visa application is being processed. A BVB must be applied for before leaving Australia."},
    {"id":"bv004","category":"Bridging Visa","text":"Bridging Visa C (BVC) is granted to people who applied for a substantive visa while holding a BVA, BVB, or BVC. It does not allow the holder to travel outside Australia."},
    {"id":"bv005","category":"Bridging Visa","text":"Bridging Visa E (BVE) is granted to unlawful non-citizens to allow them to stay lawfully in Australia while they make arrangements to depart, apply for a substantive visa, or resolve their immigration status."},
    {"id":"bv006","category":"Bridging Visa","text":"Work rights on bridging visas depend on the type of bridging visa and the circumstances of the holder. Some bridging visa holders may apply for work permission if they can demonstrate financial hardship."},
    {"id":"bv007","category":"Bridging Visa","text":"A bridging visa comes into effect only when the substantive visa ceases. If the holder still holds a valid substantive visa, the bridging visa remains dormant until the substantive visa expires."},
    {"id":"bv008","category":"Bridging Visa","text":"Bridging visa holders who need to travel should ensure they hold a Bridging Visa B before departing Australia. Travelling on a BVA or BVC will result in the visa ceasing, and the holder may not be able to re-enter Australia."},
    # HUMANITARIAN (6)
    {"id":"rh001","category":"Humanitarian","text":"Australia's Humanitarian Program provides resettlement for people who are subject to persecution or substantial discrimination in their home country. The program has an offshore component for people outside Australia and an onshore component for people seeking asylum in Australia."},
    {"id":"rh002","category":"Humanitarian","text":"The Protection Visa (Subclass 866) is for people in Australia who are found to be refugees or who meet complementary protection criteria. Applicants must demonstrate a well-founded fear of persecution based on race, religion, nationality, political opinion, or membership of a particular social group."},
    {"id":"rh003","category":"Humanitarian","text":"The Refugee Visa (Subclass 200) is for people outside Australia who are subject to persecution and who are identified and referred by the United Nations High Commissioner for Refugees (UNHCR) for resettlement."},
    {"id":"rh004","category":"Humanitarian","text":"Safe Haven Enterprise Visa (SHEV) Subclass 790 is a temporary visa for people who arrived in Australia by boat and are found to be refugees. It provides a pathway to other visa categories if certain conditions are met, including working or studying in regional Australia."},
    {"id":"rh005","category":"Humanitarian","text":"Community Support Program (CSP) allows approved community organisations to propose refugees and humanitarian entrants for resettlement in Australia. The organisations must commit to providing settlement support to the visa holders."},
    {"id":"rh006","category":"Humanitarian","text":"Global Special Humanitarian Visa (Subclass 202) is for people outside their home country who are subject to substantial discrimination amounting to a gross violation of their human rights. Applicants must be proposed by an Australian citizen, permanent resident, or eligible organisation."},
    # NZ CITIZENS (4)
    {"id":"nz001","category":"NZ Citizens","text":"New Zealand citizens can live and work in Australia on a Special Category Visa (Subclass 444) which is automatically granted on arrival. This visa allows unlimited stay and full work rights but does not provide the same benefits as permanent residency."},
    {"id":"nz002","category":"NZ Citizens","text":"The New Zealand Citizen Family Relationship Visa (Subclass 461) allows non-NZ citizen family members of NZ citizens to live and work in Australia for up to 5 years. Applicants must be the partner, dependent child, or parent of an NZ citizen."},
    {"id":"nz003","category":"NZ Citizens","text":"New Zealand citizens who have been living in Australia since before 26 February 2001 may be treated as protected SCV holders and may be eligible for Australian citizenship without first obtaining permanent residency."},
    {"id":"nz004","category":"NZ Citizens","text":"The NZ Stream of the Skilled Independent Visa (Subclass 189) provides a pathway to permanent residency for NZ citizens who meet income requirements (generally AUD 53,900 per year or above) and have lived in Australia for a specified period."},
]

print(f"Knowledge base: {len(knowledge_base)} passages")
cats = Counter(p['category'] for p in knowledge_base)
for c, n in cats.most_common():
    print(f"  {c}: {n}")

### 4.2 Core Hallucination Dataset

The core dataset evaluates whether an answer is supported by its source passage. Hallucinated examples are constructed through controlled changes such as incorrect numbers, unsupported entities, scope changes, or direct contradictions.


In [ ]:
# ============================================================
# 6.1 — Hallucination evaluation dataset (78 pairs)
# ============================================================
hallucination_dataset = [
    # GROUNDED (40)
    {"source":"The Student Visa Subclass 500 allows international students to study full-time in Australia at a recognised education institution. The visa is valid for up to 5 years depending on the course duration. Applicants must be at least 6 years old to apply.","answer":"The Student Visa 500 lets international students study full-time in Australia for up to 5 years. You need to be at least 6 years old.","label":1},
    {"source":"Student visa holders can work up to 48 hours per fortnight while their course is in session. Work rights commence once the course has started, not when the visa is granted. During scheduled course breaks, students may work unlimited hours.","answer":"Students on a student visa can work a maximum of 48 hours every two weeks while studying. They can work unlimited hours during course breaks.","label":1},
    {"source":"All student visa holders must maintain Overseas Student Health Cover (OSHC) for the entire duration of their stay in Australia. Maintaining OSHC is a mandatory visa condition, and failure to do so may result in visa cancellation.","answer":"OSHC is mandatory for all student visa holders for their entire stay. Not maintaining OSHC can lead to visa cancellation.","label":1},
    {"source":"Student visa applicants must satisfy the Genuine Student (GS) requirement, which replaced the previous Genuine Temporary Entrant (GTE) requirement.","answer":"The Genuine Student requirement has replaced the GTE requirement for student visa applicants.","label":1},
    {"source":"The base application fee for the Student Visa Subclass 500 is AUD 1,600. Additional charges apply for family members included in the application. Applicants must also demonstrate sufficient funds for living expenses, currently set at AUD 29,710 per year.","answer":"The student visa costs AUD 1,600 to apply. You also need to show you have AUD 29,710 per year for living expenses.","label":1},
    {"source":"After completing studies in Australia, students may be eligible for a Temporary Graduate Visa (Subclass 485), which allows them to work in Australia temporarily to gain practical experience in their field of study.","answer":"Graduates may be able to apply for a Subclass 485 visa to work temporarily in Australia after finishing their studies.","label":1},
    {"source":"Student visa applicants must provide proof of English language proficiency through approved tests such as IELTS, TOEFL iBT, PTE Academic, or Cambridge English.","answer":"You can prove English proficiency through IELTS, TOEFL iBT, PTE Academic, or Cambridge English.","label":1},
    {"source":"Students can apply to study two or more courses on a single Student Visa 500 where there is clear progression from one course to another. This is known as course packaging. Course gaps must be less than two calendar months between packaged courses.","answer":"Course packaging allows students to study multiple courses on one visa, as long as there is a clear progression and gaps are under two months.","label":1},
    {"source":"The Visitor Visa Subclass 600 is for people who want to visit Australia temporarily for tourism, to visit family and friends, or for business visitor activities. The visa can be granted for stays of 3, 6, or 12 months.","answer":"The Visitor Visa 600 covers tourism, family visits, and business visits for periods of 3, 6, or 12 months.","label":1},
    {"source":"Visitor visa holders are generally not permitted to work in Australia.","answer":"You cannot work on a visitor visa. It is only for tourism, seeing family, or attending business meetings.","label":1},
    {"source":"Visitor visa holders cannot study for more than 3 months on their visa.","answer":"Visitor visa holders may study for up to 3 months. Longer study requires a student visa.","label":1},
    {"source":"The Electronic Travel Authority (ETA) Subclass 601 is available for passport holders from certain eligible countries.","answer":"The ETA Subclass 601 lets eligible passport holders visit Australia for up to 3 months at a time, valid for 12 months.","label":1},
    {"source":"The Partner Visa allows the spouse or de facto partner of an Australian citizen, permanent resident, or eligible New Zealand citizen to live in Australia.","answer":"The Partner Visa is for spouses or de facto partners of Australian citizens, permanent residents, or eligible NZ citizens.","label":1},
    {"source":"The combined application charge for the Partner Visa is AUD 9,095.","answer":"The Partner Visa application costs AUD 9,095 in total.","label":1},
    {"source":"De facto relationships must have existed for at least 12 months before the visa application is lodged.","answer":"De facto couples must have been in a relationship for at least 12 months before applying, unless special circumstances apply.","label":1},
    {"source":"Partner visa holders receive work and study rights in Australia while on the temporary partner visa. They also receive access to Medicare from the date of visa grant.","answer":"Temporary partner visa holders can work and study, and they get Medicare access from the day the visa is granted.","label":1},
    {"source":"To be eligible for Australian citizenship by conferral, applicants generally must be permanent residents, have lived in Australia for at least four years including at least 12 months as a permanent resident, and be of good character.","answer":"You need to be a permanent resident who has lived in Australia for at least 4 years, with 12 months as a PR, and be of good character.","label":1},
    {"source":"Australian citizenship applicants must pass a citizenship test that assesses their knowledge of Australia. The test consists of 20 multiple-choice questions with a pass mark of 75 percent.","answer":"The citizenship test has 20 multiple-choice questions in English, and you need to score at least 75% to pass.","label":1},
    {"source":"Dual citizenship is permitted in Australia. Applicants are not required to renounce their existing citizenship.","answer":"Australia allows dual citizenship, so you do not have to give up your current citizenship.","label":1},
    {"source":"The application fee for Australian citizenship by conferral is AUD 490.","answer":"Citizenship by conferral costs AUD 490 to apply.","label":1},
    {"source":"The Skilled Independent Visa (Subclass 189) is a points-tested permanent visa. Applicants must meet the minimum points score of 65.","answer":"The Subclass 189 visa is a points-tested permanent visa. You need at least 65 points and an invitation through SkillSelect.","label":1},
    {"source":"The Working Holiday Visa (Subclass 417) allows young adults aged 18 to 30 from eligible countries to holiday and work in Australia for up to 12 months.","answer":"The Working Holiday Visa 417 is for 18-30 year olds from eligible countries, lasting up to 12 months.","label":1},
    {"source":"The Global Talent Visa (Subclass 858) is for highly skilled professionals who can demonstrate international recognition in their field.","answer":"The Global Talent Visa is designed for professionals with international recognition in their field.","label":1},
    {"source":"Most visa applications must be submitted online through ImmiAccount.","answer":"Visa applications are generally submitted online via ImmiAccount.","label":1},
    {"source":"All documents not in English must be translated by accredited translators. Translators must be accredited by NAATI.","answer":"Non-English documents need to be translated by NAATI-accredited translators.","label":1},
    {"source":"If a visa application is refused, the applicant may have the right to seek review at the Administrative Review Tribunal (ART).","answer":"You can apply for a review at the Administrative Review Tribunal if your visa is refused.","label":1},
    {"source":"Only registered migration agents and certain legal practitioners are authorised to provide immigration assistance for a fee.","answer":"Immigration assistance for a fee can only be provided by registered migration agents or legal practitioners.","label":1},
    {"source":"Visa applicants may need to undergo health examinations. Health examination results are usually valid for 12 months.","answer":"Health exams for visa applications are done by Bupa panel physicians and the results last 12 months.","label":1},
    {"source":"Bridging Visa A (BVA) is automatically granted to visa applicants who hold a valid substantive visa when they apply for a new visa in Australia.","answer":"A BVA is automatically given when you apply for a new visa while holding a valid visa in Australia.","label":1},
    {"source":"New Zealand citizens can live and work in Australia on a Special Category Visa (Subclass 444) which is automatically granted on arrival.","answer":"NZ citizens get a Special Category Visa 444 automatically when they arrive, allowing them to live and work.","label":1},
    {"source":"Australia's Humanitarian Program provides resettlement for people who are subject to persecution or substantial discrimination in their home country.","answer":"The Humanitarian Program helps resettle people facing persecution or major discrimination in their home country.","label":1},
    {"source":"Breaching visa conditions may result in visa cancellation. Common breaches include working more hours than permitted, not maintaining required health insurance.","answer":"Your visa may be cancelled for working too many hours, not keeping health insurance, or dropping out of your course.","label":1},
    {"source":"VEVO allows visa holders, employers, education providers to check visa details and conditions.","answer":"VEVO lets visa holders and employers check visa details and conditions online.","label":1},
    {"source":"Bridging Visa B (BVB) allows the holder to travel outside Australia and return while their visa application is being processed.","answer":"A Bridging Visa B allows you to leave and re-enter Australia while your visa application is being processed.","label":1},
    {"source":"The Parent Visa (Subclass 103) allows parents to live in Australia permanently. Processing times often exceed 30 years.","answer":"The Parent Visa 103 gives parents permanent residency but can take over 30 years to process.","label":1},
    {"source":"Children born in Australia to at least one parent who is an Australian citizen or permanent resident automatically acquire Australian citizenship.","answer":"If at least one parent is an Australian citizen or PR, their child born in Australia automatically gets citizenship.","label":1},
    {"source":"Condition 8105 limits the holder to working no more than 48 hours per fortnight when the holder's course of study is in session.","answer":"Visa condition 8105 restricts work to 48 hours per fortnight during study sessions.","label":1},
    {"source":"The Contributory Parent Visa (Subclass 143) has a second instalment of approximately AUD 47,755. Processing times are typically 5 to 7 years.","answer":"The Contributory Parent Visa costs about AUD 47,755 as a second instalment and takes 5 to 7 years to process.","label":1},
    {"source":"The Skilled Nominated Visa (Subclass 190) is a points-tested permanent visa. Nomination adds 5 points.","answer":"The Subclass 190 visa gives skilled workers permanent residency through state nomination, which adds 5 points.","label":1},
    {"source":"Temporary Graduate Visa (Subclass 485) Post-Study Work stream validity ranges from 2 to 6 years depending on the qualification.","answer":"The Post-Study Work stream of the 485 visa lasts between 2 and 6 years based on your qualification.","label":1},

    # HALLUCINATED (38)
    {"source":"The Student Visa Subclass 500 allows international students to study full-time in Australia. The visa is valid for up to 5 years. Applicants must be at least 6 years old.","answer":"The Student Visa 500 allows both full-time and part-time study. The visa is valid for up to 10 years and there is no minimum age requirement.","label":0},
    {"source":"Student visa holders can work up to 48 hours per fortnight while their course is in session. Work rights commence once the course has started.","answer":"Student visa holders can work unlimited hours during their course. Work rights begin immediately when the visa is issued, even before the course starts.","label":0},
    {"source":"All student visa holders must maintain Overseas Student Health Cover (OSHC) for the entire duration of their stay.","answer":"OSHC is optional for student visa holders. Students can instead use Australia's Medicare system for free healthcare.","label":0},
    {"source":"The base application fee for the Student Visa Subclass 500 is AUD 1,600.","answer":"The Student Visa costs AUD 500 to apply, making it one of the cheapest visa categories in Australia.","label":0},
    {"source":"Student visa applicants must provide proof of English language proficiency through approved tests such as IELTS, TOEFL iBT, PTE Academic, or Cambridge English. The Department does not accept results from tests taken at home or online.","answer":"Students can prove English proficiency through any online test including Duolingo, TOEFL Home Edition, and IELTS Online. No in-person testing is required.","label":0},
    {"source":"For 2026, Australia has implemented a National Planning Level (NPL) cap of 295,000 international student places.","answer":"Australia has removed all caps on international students for 2026, with unlimited places available at all institutions.","label":0},
    {"source":"Student visa applicants must satisfy the Genuine Student (GS) requirement.","answer":"The GS requirement was abolished in 2025 and students no longer need to prove genuine intent to study.","label":0},
    {"source":"After completing studies in Australia, students may be eligible for a Temporary Graduate Visa (Subclass 485).","answer":"All students automatically receive permanent residency upon completing their studies in Australia.","label":0},
    {"source":"The Visitor Visa Subclass 600 is for people who want to visit Australia temporarily. The visa can be granted for stays of 3, 6, or 12 months.","answer":"The Visitor Visa 600 allows you to work part-time in Australia and can be granted for up to 3 years.","label":0},
    {"source":"Visitor visa holders are generally not permitted to work in Australia.","answer":"Visitor visa holders can work up to 20 hours per week in Australia without any additional permits.","label":0},
    {"source":"Visitor visa holders cannot study for more than 3 months on their visa.","answer":"Visitor visa holders can study for up to 2 years in Australia without needing a student visa.","label":0},
    {"source":"The Electronic Travel Authority (ETA) Subclass 601 is available for passport holders from certain eligible countries.","answer":"The ETA is available to citizens of all countries worldwide and provides automatic permanent residency.","label":0},
    {"source":"The Partner Visa allows the spouse or de facto partner of an Australian citizen, permanent resident, or eligible New Zealand citizen to live in Australia.","answer":"The Partner Visa is available to friends, distant relatives, and ex-partners of Australian citizens. No romantic relationship is required.","label":0},
    {"source":"The combined application charge for the Partner Visa is AUD 9,095.","answer":"The Partner Visa is free of charge for all applicants. The Australian government subsidises 100% of the application fee.","label":0},
    {"source":"De facto relationships must have existed for at least 12 months before the visa application is lodged.","answer":"De facto couples can apply immediately after starting their relationship. There is no minimum relationship duration.","label":0},
    {"source":"Partner visa sponsors must be approved by the Department. Sponsors may be refused if they have relevant criminal convictions.","answer":"Anyone can sponsor a partner visa regardless of their criminal history. The Department does not check sponsors' backgrounds.","label":0},
    {"source":"To be eligible for Australian citizenship by conferral, applicants generally must be permanent residents, have lived in Australia for at least four years.","answer":"Anyone who has visited Australia once can apply for citizenship. There is no residency requirement at all.","label":0},
    {"source":"Australian citizenship applicants must pass a citizenship test. The test consists of 20 multiple-choice questions with a pass mark of 75 percent.","answer":"The citizenship test can be taken in any language, has 5 questions, and requires only 30% to pass.","label":0},
    {"source":"Dual citizenship is permitted in Australia. Applicants are not required to renounce their existing citizenship.","answer":"Australia does not allow dual citizenship. You must renounce all other citizenships before becoming Australian.","label":0},
    {"source":"The application fee for Australian citizenship by conferral is AUD 490.","answer":"Citizenship costs AUD 5,000 to apply and an additional AUD 2,000 for the ceremony fee.","label":0},
    {"source":"The Skilled Independent Visa (Subclass 189) is a points-tested permanent visa. Applicants must meet the minimum points score of 65.","answer":"The Subclass 189 visa does not require any points. Anyone with a job offer can apply directly.","label":0},
    {"source":"The Working Holiday Visa (Subclass 417) allows young adults aged 18 to 30 from eligible countries to work in Australia for up to 12 months.","answer":"The Working Holiday Visa has no age restriction and allows stays of up to 5 years. All nationalities can apply.","label":0},
    {"source":"The Temporary Skill Shortage Visa (Subclass 482) allows employers to sponsor skilled overseas workers.","answer":"The Subclass 482 visa does not require employer sponsorship. Workers can self-nominate for any occupation.","label":0},
    {"source":"The Global Talent Visa (Subclass 858) is for highly skilled professionals.","answer":"The Global Talent Visa is for entry-level workers and requires no special skills or recognition.","label":0},
    {"source":"Most visa applications must be submitted online through ImmiAccount.","answer":"All visa applications must be submitted by mail to the nearest Australian embassy. Online applications are not accepted.","label":0},
    {"source":"All documents not in English must be translated by accredited translators. Translators must be accredited by NAATI.","answer":"Documents can be translated by anyone including family members and friends. No accreditation is needed.","label":0},
    {"source":"If a visa application is refused, the applicant may have the right to seek review at the Administrative Review Tribunal (ART).","answer":"If your visa is refused, the decision is completely final. There is no right of appeal or review under any circumstances.","label":0},
    {"source":"Only registered migration agents and certain legal practitioners are authorised to provide immigration assistance for a fee.","answer":"Anyone can charge a fee for immigration advice in Australia. There are no licensing or registration requirements.","label":0},
    {"source":"Visa applicants may need to undergo health examinations. Results are usually valid for 12 months.","answer":"Health examinations are never required for any Australian visa application. This requirement was removed in 2024.","label":0},
    {"source":"Applicants must meet character requirements for most Australian visas. This includes providing police clearance certificates.","answer":"Character checks are not required for any visa. Australia does not request police clearances from any applicant.","label":0},
    {"source":"Bridging Visa A (BVA) is automatically granted to visa applicants who hold a valid substantive visa when they apply for a new visa.","answer":"Bridging Visa A provides permanent residency rights and is granted to all tourists on arrival.","label":0},
    {"source":"Bridging Visa B (BVB) allows the holder to travel outside Australia and return while their visa application is being processed.","answer":"Bridging Visa B is automatically granted to everyone. You don't need to apply for it and it allows unlimited international travel.","label":0},
    {"source":"New Zealand citizens can live and work in Australia on a Special Category Visa (Subclass 444) which is automatically granted on arrival.","answer":"New Zealand citizens must apply for a full work visa before entering Australia. No automatic visa is available.","label":0},
    {"source":"The Parent Visa (Subclass 103) allows parents to live in Australia permanently. Processing times often exceed 30 years.","answer":"The Parent Visa is processed within 2 weeks and costs nothing to apply for.","label":0},
    {"source":"The Contributory Parent Visa (Subclass 143) has a second instalment of approximately AUD 47,755.","answer":"The Contributory Parent Visa is completely free. All costs are covered by the Australian government.","label":0},
    {"source":"VEVO allows visa holders, employers, education providers to check visa details and conditions. VEVO shows details relating to the current in-effect visa only.","answer":"VEVO allows anyone to access any person's complete immigration history including all travel records and personal information.","label":0},
    {"source":"Children born in Australia to at least one parent who is an Australian citizen or permanent resident automatically acquire Australian citizenship.","answer":"Children born in Australia never receive automatic citizenship regardless of their parents' status. They must apply at age 21.","label":0},
    {"source":"Condition 8105 limits the holder to working no more than 48 hours per fortnight when the holder's course is in session.","answer":"Condition 8105 allows students to work 80 hours per week with no restrictions during study periods.","label":0},
]

grounded = sum(1 for d in hallucination_dataset if d['label'] == 1)
hallucinated = sum(1 for d in hallucination_dataset if d['label'] == 0)
print(f"Dataset: {len(hallucination_dataset)} samples ({grounded} grounded, {hallucinated} hallucinated)")

### 4.3 Leakage-Free Splitting

The split is performed before augmentation. This matters because augmenting first could place near-identical variants of the same answer in both train and test sets, inflating performance.


In [ ]:
# ============================================================
# CRITICAL: Split BEFORE augmentation to prevent leakage
# ============================================================
def prepare_dataset(dataset, test_size=0.3, augment_multiplier=4, seed=42):
    """
    Split dataset into train/test, then augment ONLY training set.
    Returns: train_data (augmented), test_data (original only)
    """
    labels = [d['label'] for d in dataset]

    # Stratified split on ORIGINAL data
    train_raw, test_raw = train_test_split(
        dataset, test_size=test_size, random_state=seed, stratify=labels
    )

    # Augment training set only
    train_augmented = list(train_raw)
    for _ in range(augment_multiplier - 1):
        for item in train_raw:
            words = item['answer'].split()
            if len(words) > 5:
                i = random.randint(0, len(words) - 2)
                words[i], words[i + 1] = words[i + 1], words[i]
            train_augmented.append({
                'source': item['source'],
                'answer': ' '.join(words),
                'label': item['label']
            })

    print(f"Original: {len(dataset)} | Train raw: {len(train_raw)} | "
          f"Train augmented: {len(train_augmented)} | Test: {len(test_raw)}")
    print(f"Test set contains ONLY original samples (no leakage).")

    return train_augmented, test_raw

train_data, test_data = prepare_dataset(hallucination_dataset)

### 4.4 v5 Hard-Negative Expansion

The v5 dataset extends the original examples with more subtle hallucinations. Difficulty labels support category-wise analysis, making it possible to identify whether the system struggles more with numerical changes, entity confusion, paraphrase, or unsupported inference.


In [ ]:
# ════════════════════════════════════════════════════════════════
# U1 — EXPANDED DATASET WITH HARD NEGATIVES + DIFFICULTY LABELS
# ════════════════════════════════════════════════════════════════
# Paste this cell AFTER Section 6.1 in v4 (or replace hallucination_dataset)

# fmt: off
hallucination_dataset_v5 = [
    # ──────────────────────────────────────────────────────────
    # GROUNDED — Original core (40) + 20 new
    # ──────────────────────────────────────────────────────────
    # --- Original 40 grounded (carry over from v4, add 'difficulty' key) ---
    {"source":"The Student Visa Subclass 500 allows international students to study full-time in Australia at a registered institution.","answer":"The Student Visa 500 lets international students study full-time at registered Australian institutions.","label":1,"difficulty":"paraphrase"},
    {"source":"Student visa holders can work up to 48 hours per fortnight while their course is in session. Work is unlimited during scheduled breaks.","answer":"Students on a student visa can work a maximum of 48 hours per fortnight during term time, with unlimited hours during breaks.","label":1,"difficulty":"paraphrase"},
    {"source":"All student visa holders must maintain Overseas Student Health Cover (OSHC) for the entire duration of their stay.","answer":"OSHC must be maintained by student visa holders throughout their entire stay in Australia.","label":1,"difficulty":"paraphrase"},
    {"source":"Student visa applicants must satisfy the Genuine Student (GS) requirement, which replaced the previous GTE requirement in 2024.","answer":"The Genuine Student requirement, which replaced GTE in 2024, must be satisfied by student visa applicants.","label":1,"difficulty":"paraphrase"},
    {"source":"The base application fee for the Student Visa Subclass 500 is AUD 1,600. Additional charges apply for family members.","answer":"A Student Visa 500 application costs AUD 1,600 as the base fee, with extra charges for dependants.","label":1,"difficulty":"numerical"},
    {"source":"After completing studies in Australia, students may be eligible for a Temporary Graduate Visa (Subclass 485).","answer":"Graduates may apply for the Subclass 485 Temporary Graduate Visa after finishing their Australian studies.","label":1,"difficulty":"paraphrase"},
    {"source":"Student visa applicants must provide proof of English language proficiency through approved tests such as IELTS, TOEFL, or PTE.","answer":"English proficiency must be proven by student visa applicants via tests like IELTS, TOEFL, or PTE.","label":1,"difficulty":"paraphrase"},
    {"source":"Students can apply to study two or more courses on a single Student Visa 500 where there is clear academic progression.","answer":"Multiple courses can be covered under one Student Visa 500 if clear academic progression exists.","label":1,"difficulty":"paraphrase"},
    {"source":"The Visitor Visa Subclass 600 is for people who want to visit Australia temporarily for tourism, business, or family visits.","answer":"The Subclass 600 Visitor Visa covers temporary visits for tourism, business, or family purposes.","label":1,"difficulty":"paraphrase"},
    {"source":"Visitor visa holders are generally not permitted to work in Australia.","answer":"You cannot work in Australia on a visitor visa.","label":1,"difficulty":"lexical"},
    {"source":"Visitor visa holders cannot study for more than 3 months on their visa.","answer":"Visitor visa holders are limited to 3 months of study.","label":1,"difficulty":"numerical"},
    {"source":"The Electronic Travel Authority (ETA) Subclass 601 is available for passport holders from certain countries.","answer":"The ETA Subclass 601 is available to citizens of specific countries.","label":1,"difficulty":"lexical"},
    {"source":"The Partner Visa allows the spouse or de facto partner of an Australian citizen, permanent resident, or eligible New Zealand citizen to live in Australia.","answer":"Partners of Australian citizens, permanent residents, or eligible NZ citizens can apply for the Partner Visa to live in Australia.","label":1,"difficulty":"paraphrase"},
    {"source":"The combined application charge for the Partner Visa is AUD 9,095.","answer":"The Partner Visa application fee totals AUD 9,095.","label":1,"difficulty":"numerical"},
    {"source":"De facto relationships must have existed for at least 12 months before the visa application is lodged.","answer":"A de facto relationship needs to have been in place for a minimum of 12 months prior to lodging the visa application.","label":1,"difficulty":"numerical"},
    {"source":"Partner visa holders receive work and study rights in Australia while on the temporary partner visa.","answer":"Temporary partner visa holders can work and study in Australia.","label":1,"difficulty":"lexical"},
    {"source":"To be eligible for Australian citizenship by conferral, applicants generally must be permanent residents who have lived in Australia for at least 4 years.","answer":"Citizenship by conferral generally requires at least 4 years of residence as a permanent resident.","label":1,"difficulty":"numerical"},
    {"source":"Australian citizenship applicants must pass a citizenship test that assesses their knowledge of Australia, its values, history, and national symbols.","answer":"The citizenship test covers Australian values, history, and national symbols.","label":1,"difficulty":"paraphrase"},
    {"source":"Dual citizenship is permitted in Australia. Applicants are not required to renounce their existing citizenship.","answer":"Australia allows dual citizenship without requiring renunciation of other nationalities.","label":1,"difficulty":"lexical"},
    {"source":"The application fee for Australian citizenship by conferral is AUD 490.","answer":"Citizenship by conferral costs AUD 490.","label":1,"difficulty":"numerical"},
    {"source":"The Skilled Independent Visa (Subclass 189) is a points-tested permanent visa. Applicants must meet the minimum points threshold of 65.","answer":"The Subclass 189 visa is a points-tested permanent visa requiring a minimum of 65 points.","label":1,"difficulty":"numerical"},
    {"source":"The Working Holiday Visa (Subclass 417) allows young adults aged 18 to 30 from eligible countries to work and travel in Australia for up to 12 months.","answer":"Young people aged 18–30 from eligible countries can work and travel in Australia for up to 12 months on the Subclass 417 visa.","label":1,"difficulty":"numerical"},
    {"source":"The Global Talent Visa (Subclass 858) is for highly skilled professionals who can demonstrate international recognition in their field.","answer":"The Subclass 858 Global Talent Visa targets internationally recognised professionals.","label":1,"difficulty":"lexical"},
    {"source":"Most visa applications must be submitted online through ImmiAccount.","answer":"Visa applications are generally submitted online via ImmiAccount.","label":1,"difficulty":"lexical"},
    {"source":"All documents not in English must be translated by accredited translators. Translators must be accredited by NAATI.","answer":"Non-English documents require translation by NAATI-accredited translators.","label":1,"difficulty":"paraphrase"},
    {"source":"If a visa application is refused, the applicant may have the right to seek review at the Administrative Appeals Tribunal (AAT).","answer":"Refused visa applicants may appeal to the AAT.","label":1,"difficulty":"lexical"},
    {"source":"Only registered migration agents and certain legal practitioners are authorised to provide immigration assistance for a fee.","answer":"Paid immigration advice can only be provided by registered migration agents or qualified legal practitioners.","label":1,"difficulty":"paraphrase"},
    {"source":"Visa applicants may need to undergo health examinations. Health examination results are usually valid for 12 months.","answer":"Health exam results for visa applications are typically valid for 12 months.","label":1,"difficulty":"numerical"},
    {"source":"Bridging Visa A (BVA) is automatically granted to visa applicants who hold a valid substantive visa when they apply for a new visa in Australia.","answer":"BVA is automatically given when a person holding a valid substantive visa applies for a new one in Australia.","label":1,"difficulty":"paraphrase"},
    {"source":"New Zealand citizens can live and work in Australia on a Special Category Visa (Subclass 444) which is automatically granted on arrival.","answer":"NZ citizens automatically receive a Subclass 444 Special Category Visa on arrival, allowing them to live and work.","label":1,"difficulty":"paraphrase"},
    {"source":"Australia's Humanitarian Program provides resettlement for people who are subject to persecution or violence in their home countries.","answer":"The Humanitarian Program resettles people facing persecution or violence.","label":1,"difficulty":"lexical"},
    {"source":"Breaching visa conditions may result in visa cancellation. Common breaches include working more hours than permitted.","answer":"Working beyond permitted hours is a common visa breach that can lead to cancellation.","label":1,"difficulty":"paraphrase"},
    {"source":"VEVO allows visa holders, employers, education providers to check visa details and conditions.","answer":"Visa details and conditions can be checked via VEVO by holders, employers, and education providers.","label":1,"difficulty":"lexical"},
    {"source":"Bridging Visa B (BVB) allows the holder to travel outside Australia and return while their visa application is being processed.","answer":"BVB holders can leave and re-enter Australia while awaiting visa processing.","label":1,"difficulty":"paraphrase"},
    {"source":"The Parent Visa (Subclass 103) allows parents to live in Australia permanently. Processing times can exceed 30 years due to high demand.","answer":"The Subclass 103 Parent Visa grants permanent residence, but processing can take over 30 years.","label":1,"difficulty":"numerical"},
    {"source":"Children born in Australia to at least one parent who is an Australian citizen or permanent resident automatically acquire Australian citizenship.","answer":"A child born in Australia to a citizen or permanent resident parent is automatically an Australian citizen.","label":1,"difficulty":"paraphrase"},
    {"source":"Condition 8105 limits the holder to working no more than 48 hours per fortnight when the holder's course is in session.","answer":"Condition 8105 restricts work to 48 hours per fortnight during the academic session.","label":1,"difficulty":"numerical"},
    {"source":"The Contributory Parent Visa (Subclass 143) has a second instalment of approximately AUD 47,755. Processing times are faster than the regular Parent Visa.","answer":"The Subclass 143 visa costs approximately AUD 47,755 as a second instalment and is processed faster than the Subclass 103.","label":1,"difficulty":"numerical"},
    {"source":"The Skilled Nominated Visa (Subclass 190) is a points-tested permanent visa. Nomination adds 5 points to the applicant's score.","answer":"State nomination under the Subclass 190 visa adds 5 points to the applicant's points test score.","label":1,"difficulty":"numerical"},
    {"source":"Temporary Graduate Visa (Subclass 485) Post-Study Work stream validity ranges from 2 to 6 years depending on qualification level.","answer":"The Subclass 485 Post-Study Work stream lasts between 2 and 6 years based on the qualification obtained.","label":1,"difficulty":"numerical"},

    # --- NEW grounded examples (20) ---
    {"source":"The Temporary Skill Shortage Visa (Subclass 482) allows employers to sponsor overseas workers for up to 4 years.","answer":"Employers can sponsor skilled overseas workers for up to 4 years under the Subclass 482 visa.","label":1,"difficulty":"numerical"},
    {"source":"The Employer Nomination Scheme Visa (Subclass 186) is a permanent visa for skilled workers nominated by their employer.","answer":"The Subclass 186 is a permanent visa for employer-nominated skilled workers.","label":1,"difficulty":"lexical"},
    {"source":"Points-tested skilled visas award points for factors including age, English proficiency, qualifications, and work experience.","answer":"Age, English ability, qualifications, and work experience all earn points for skilled visa applications.","label":1,"difficulty":"paraphrase"},
    {"source":"The Training Visa (Subclass 407) allows people to participate in occupational training to improve their skills for their current occupation.","answer":"The Subclass 407 Training Visa is for occupational skills improvement through training.","label":1,"difficulty":"paraphrase"},
    {"source":"The Skilled Work Regional Visa (Subclass 491) is a points-tested provisional visa valid for 5 years.","answer":"The Subclass 491 visa is a provisional, points-tested regional visa valid for 5 years.","label":1,"difficulty":"numerical"},
    {"source":"Working Holiday Visa holders who perform specified work in regional areas may be eligible for a second or third year visa.","answer":"Regional specified work can qualify Working Holiday Visa holders for second or third year extensions.","label":1,"difficulty":"paraphrase"},
    {"source":"The Protection Visa (Subclass 866) is for people in Australia who are recognised as refugees under the Refugees Convention.","answer":"People in Australia who meet the Refugees Convention definition can apply for the Subclass 866 Protection Visa.","label":1,"difficulty":"paraphrase"},
    {"source":"The general residence requirement for citizenship is four years as a permanent resident, including at least 12 months on a permanent visa.","answer":"Citizenship requires four years of residence, with at least 12 months on a permanent visa.","label":1,"difficulty":"numerical"},
    {"source":"Visa Application Charge (VAC) refunds are available only if the application is withdrawn before a decision is made, and the refund amount varies.","answer":"VAC refunds may be given if the application is withdrawn before decision, with the amount varying.","label":1,"difficulty":"paraphrase"},
    {"source":"Ministerial Direction 115 sets out the order of priority for processing visa applications.","answer":"Visa processing priorities are governed by Ministerial Direction 115.","label":1,"difficulty":"lexical"},
    {"source":"Student visa holders who wish to change their education provider must notify the Department and may need a new CoE.","answer":"Changing education providers requires notifying the Department and potentially obtaining a new CoE.","label":1,"difficulty":"paraphrase"},
    {"source":"The Prospective Marriage Visa (Subclass 300) allows the fiance of an Australian citizen or permanent resident to travel to Australia and marry within 9 months.","answer":"Fiances of Australian citizens or permanent residents can come to Australia and must marry within 9 months on the Subclass 300 visa.","label":1,"difficulty":"numerical"},
    {"source":"Condition 8202 requires student visa holders to remain enrolled in and attend their registered course.","answer":"Under Condition 8202, students must stay enrolled in and attend their registered course.","label":1,"difficulty":"lexical"},
    {"source":"Condition 8501 requires the visa holder to maintain adequate health insurance.","answer":"Health insurance must be maintained under Condition 8501.","label":1,"difficulty":"lexical"},
    {"source":"Health waivers may be granted in certain circumstances where the applicant does not meet the health requirement.","answer":"Applicants who fail the health requirement may be granted a waiver in certain cases.","label":1,"difficulty":"paraphrase"},
    {"source":"The citizenship test consists of 20 multiple-choice questions in English, and applicants must score at least 75% to pass.","answer":"The citizenship test has 20 multiple-choice questions and requires a 75% pass mark.","label":1,"difficulty":"numerical"},
    {"source":"The Safe Haven Enterprise Visa (SHEV) Subclass 790 is a temporary visa for up to 5 years.","answer":"The SHEV (Subclass 790) provides temporary protection for up to 5 years.","label":1,"difficulty":"numerical"},
    {"source":"The NZ Stream of the Skilled Independent Visa (Subclass 189) allows eligible NZ citizens to apply for permanent residency.","answer":"Eligible NZ citizens can gain permanent residency through the NZ Stream of the Subclass 189 visa.","label":1,"difficulty":"paraphrase"},
    {"source":"Community Support Program (CSP) allows approved community organisations to propose refugees for resettlement.","answer":"Approved community organisations can propose refugees for resettlement under the CSP.","label":1,"difficulty":"lexical"},
    {"source":"Financial evidence for a student visa must demonstrate access to at least AUD 29,710 per year for living costs.","answer":"Student visa applicants must show financial evidence of at least AUD 29,710 per year for living expenses.","label":1,"difficulty":"numerical"},

    # ──────────────────────────────────────────────────────────
    # HALLUCINATED — Original (38) + 22 new hard negatives
    # ──────────────────────────────────────────────────────────
    # --- Original 38 hallucinated (carry over from v4, add 'difficulty') ---
    {"source":"The Student Visa Subclass 500 allows international students to study full-time in Australia. The visa is valid for the duration of the enrolled course plus additional months.","answer":"The Student Visa 500 allows students to study part-time or full-time at any Australian institution, including unregistered providers.","label":0,"difficulty":"semantic"},
    {"source":"Student visa holders can work up to 48 hours per fortnight while their course is in session. Work is unlimited during scheduled breaks.","answer":"Student visa holders are allowed to work up to 48 hours per week with no restrictions during any period.","label":0,"difficulty":"numerical"},
    {"source":"All student visa holders must maintain Overseas Student Health Cover (OSHC) for the entire duration of their stay.","answer":"OSHC is recommended but not mandatory for student visa holders. Students can choose any health insurance provider.","label":0,"difficulty":"semantic"},
    {"source":"The base application fee for the Student Visa Subclass 500 is AUD 1,600.","answer":"The Student Visa 500 application fee is AUD 800, making it one of the most affordable visa categories.","label":0,"difficulty":"numerical"},
    {"source":"Student visa applicants must provide proof of English language proficiency through approved tests such as IELTS, TOEFL, or PTE.","answer":"English proficiency is not required for student visa applications. Students can demonstrate language skills after arriving in Australia.","label":0,"difficulty":"semantic"},
    {"source":"For 2026, Australia has implemented a National Planning Level (NPL) cap of 295,000 international students.","answer":"Australia has no cap on international student numbers and accepts unlimited applications each year.","label":0,"difficulty":"semantic"},
    {"source":"Student visa applicants must satisfy the Genuine Student (GS) requirement.","answer":"The GS requirement was abolished in 2025 and is no longer part of the student visa process.","label":0,"difficulty":"semantic"},
    {"source":"After completing studies in Australia, students may be eligible for a Temporary Graduate Visa (Subclass 485).","answer":"All graduates automatically receive permanent residency upon completing their Australian qualifications.","label":0,"difficulty":"semantic"},
    {"source":"The Visitor Visa Subclass 600 is for people who want to visit Australia temporarily. The visa can be granted for 3, 6, or 12 months.","answer":"The Visitor Visa 600 allows stays of up to 5 years for all applicants regardless of their country of origin.","label":0,"difficulty":"numerical"},
    {"source":"Visitor visa holders are generally not permitted to work in Australia.","answer":"Visitor visa holders can work up to 20 hours per week in Australia during their stay.","label":0,"difficulty":"semantic"},
    {"source":"Visitor visa holders cannot study for more than 3 months on their visa.","answer":"Visitor visa holders can study for up to 12 months without needing a student visa.","label":0,"difficulty":"numerical"},
    {"source":"The Electronic Travel Authority (ETA) Subclass 601 is available for passport holders from certain countries.","answer":"The ETA is available to all countries worldwide with no restrictions on nationality.","label":0,"difficulty":"semantic"},
    {"source":"The Partner Visa allows the spouse or de facto partner of an Australian citizen, permanent resident, or eligible New Zealand citizen to live in Australia.","answer":"The Partner Visa is available to friends, colleagues, or any person who has a close relationship with an Australian citizen.","label":0,"difficulty":"semantic"},
    {"source":"The combined application charge for the Partner Visa is AUD 9,095.","answer":"The Partner Visa is free of charge for all applicants as part of Australia's family reunion initiative.","label":0,"difficulty":"numerical"},
    {"source":"De facto relationships must have existed for at least 12 months before the visa application is lodged.","answer":"De facto relationships only need to be 3 months old before a partner visa can be applied for.","label":0,"difficulty":"numerical"},
    {"source":"Partner visa sponsors must be approved by the Department. Sponsors may be refused if they have relevant criminal convictions.","answer":"Anyone can sponsor a partner visa applicant with no background checks or approval process required.","label":0,"difficulty":"semantic"},
    {"source":"To be eligible for Australian citizenship by conferral, applicants generally must be permanent residents who have lived in Australia for at least 4 years.","answer":"Australian citizenship requires only 1 year of residence in the country, regardless of visa status.","label":0,"difficulty":"numerical"},
    {"source":"Australian citizenship applicants must pass a citizenship test. The test consists of 20 multiple-choice questions in English.","answer":"The citizenship test can be taken in any language and consists of 10 true-or-false questions.","label":0,"difficulty":"numerical"},
    {"source":"Dual citizenship is permitted in Australia. Applicants are not required to renounce their existing citizenship.","answer":"Australia does not allow dual citizenship. All new citizens must renounce their previous nationality within 6 months.","label":0,"difficulty":"semantic"},
    {"source":"The application fee for Australian citizenship by conferral is AUD 490.","answer":"Citizenship costs AUD 2,500 including mandatory legal representation fees.","label":0,"difficulty":"numerical"},
    {"source":"The Skilled Independent Visa (Subclass 189) is a points-tested permanent visa. Applicants must meet the minimum points threshold of 65.","answer":"The Subclass 189 visa requires only 45 points and does not require a skills assessment.","label":0,"difficulty":"numerical"},
    {"source":"The Working Holiday Visa (Subclass 417) allows young adults aged 18 to 30 from eligible countries to work and travel in Australia for up to 12 months.","answer":"The Working Holiday Visa is available to people aged 18 to 45 from any country worldwide.","label":0,"difficulty":"numerical"},
    {"source":"The Temporary Skill Shortage Visa (Subclass 482) allows employers to sponsor skilled overseas workers.","answer":"The Subclass 482 visa allows workers to self-sponsor without any employer involvement.","label":0,"difficulty":"semantic"},
    {"source":"The Global Talent Visa (Subclass 858) is for highly skilled professionals.","answer":"The Global Talent Visa is available to anyone with a bachelor's degree and 2 years of work experience.","label":0,"difficulty":"semantic"},
    {"source":"Most visa applications must be submitted online through ImmiAccount.","answer":"All visa applications must be submitted in person at an Australian embassy or consulate. Online applications are not accepted.","label":0,"difficulty":"semantic"},
    {"source":"All documents not in English must be translated by accredited translators. Translators must be accredited by NAATI.","answer":"Documents in foreign languages can be translated by any bilingual person and do not need professional accreditation.","label":0,"difficulty":"semantic"},
    {"source":"If a visa application is refused, the applicant may have the right to seek review at the Administrative Appeals Tribunal (AAT).","answer":"Refused visa applications cannot be appealed. The decision of the Department is always final.","label":0,"difficulty":"semantic"},
    {"source":"Only registered migration agents and certain legal practitioners are authorised to provide immigration assistance for a fee.","answer":"Any person can provide paid immigration advice in Australia without any registration or qualifications.","label":0,"difficulty":"semantic"},
    {"source":"Visa applicants may need to undergo health examinations. Results are usually valid for 12 months.","answer":"Health examinations are never required for any Australian visa application.","label":0,"difficulty":"semantic"},
    {"source":"Applicants must meet character requirements for most Australian visas. This includes providing police clearance certificates.","answer":"Character requirements have been abolished for all Australian visa categories since 2024.","label":0,"difficulty":"semantic"},
    {"source":"Bridging Visa A (BVA) is automatically granted to visa applicants who hold a valid substantive visa when they apply for a new visa in Australia.","answer":"Bridging Visa A must be separately applied for and costs AUD 500 regardless of circumstances.","label":0,"difficulty":"semantic"},
    {"source":"Bridging Visa B (BVB) allows the holder to travel outside Australia and return while their visa application is being processed.","answer":"Bridging Visa B does not allow travel. Holders who leave Australia automatically have their visa cancelled.","label":0,"difficulty":"semantic"},
    {"source":"New Zealand citizens can live and work in Australia on a Special Category Visa (Subclass 444) which is automatically granted on arrival.","answer":"New Zealand citizens must apply for a work permit 6 months before travelling to Australia.","label":0,"difficulty":"semantic"},
    {"source":"The Parent Visa (Subclass 103) allows parents to live in Australia permanently. Processing times can exceed 30 years.","answer":"The Parent Visa is processed within 6 months and costs only AUD 500.","label":0,"difficulty":"numerical"},
    {"source":"The Contributory Parent Visa (Subclass 143) has a second instalment of approximately AUD 47,755.","answer":"The Contributory Parent Visa has no second instalment. The total cost is under AUD 5,000.","label":0,"difficulty":"numerical"},
    {"source":"VEVO allows visa holders, employers, education providers to check visa details and conditions. VEVO is a free online service.","answer":"VEVO requires a paid subscription of AUD 50 per month for employers to access visa holder information.","label":0,"difficulty":"numerical"},
    {"source":"Children born in Australia to at least one parent who is an Australian citizen or permanent resident automatically acquire citizenship.","answer":"Children born in Australia do not receive automatic citizenship. They must apply separately after turning 18.","label":0,"difficulty":"semantic"},
    {"source":"Condition 8105 limits the holder to working no more than 48 hours per fortnight when the holder's course is in session.","answer":"Condition 8105 allows unlimited work hours at all times during the student visa period.","label":0,"difficulty":"semantic"},

    # --- NEW hard negatives: SUBTLE hallucinations (22) ---
    # Numerical — off by a small amount
    {"source":"Student visa holders can work up to 48 hours per fortnight while their course is in session.","answer":"Students on a student visa can work up to 40 hours per fortnight during their academic semester.","label":0,"difficulty":"numerical"},
    {"source":"The base application fee for the Student Visa Subclass 500 is AUD 1,600.","answer":"The Student Visa 500 base fee is AUD 1,500, with no additional charges for dependants.","label":0,"difficulty":"numerical"},
    {"source":"The combined application charge for the Partner Visa is AUD 9,095.","answer":"The Partner Visa costs AUD 8,095 for the primary applicant.","label":0,"difficulty":"numerical"},
    {"source":"To be eligible for Australian citizenship by conferral, applicants must have lived in Australia for at least 4 years.","answer":"Citizenship by conferral requires at least 3 years of residence as a permanent resident in Australia.","label":0,"difficulty":"numerical"},
    {"source":"The Skilled Independent Visa (Subclass 189) requires a minimum points score of 65.","answer":"Applicants for the Subclass 189 visa need a minimum of 60 points on the skilled migration test.","label":0,"difficulty":"numerical"},
    {"source":"The citizenship test consists of 20 multiple-choice questions and requires a 75% pass mark.","answer":"The citizenship test has 25 multiple-choice questions with a 70% pass mark.","label":0,"difficulty":"numerical"},

    # Entity swap — correct structure but wrong entity
    {"source":"The Partner Visa allows the spouse or de facto partner of an Australian citizen to live in Australia.","answer":"The Partner Visa is available to engaged couples who plan to marry within 24 months of arriving in Australia.","label":0,"difficulty":"retrieval-confusion"},
    {"source":"Bridging Visa A (BVA) is automatically granted to visa applicants who hold a valid substantive visa.","answer":"Bridging Visa C is automatically granted to visa applicants who hold a valid substantive visa when they apply for a new one.","label":0,"difficulty":"retrieval-confusion"},
    {"source":"The Visitor Visa Subclass 600 can be granted for stays of 3, 6, or 12 months.","answer":"The Visitor Visa 600 allows stays of 3, 6, 12, or 24 months depending on the applicant's circumstances.","label":0,"difficulty":"numerical"},
    {"source":"Visa Application Charge (VAC) refunds are available only if the application is withdrawn before a decision is made.","answer":"Visa Application Charges are fully refundable at any time, including after a visa decision has been made.","label":0,"difficulty":"semantic"},

    # Temporal — wrong year or timeframe
    {"source":"Student visa applicants must satisfy the Genuine Student (GS) requirement, which replaced the previous GTE requirement in 2024.","answer":"The Genuine Student requirement replaced the GTE in 2022 as part of the broader migration reform package.","label":0,"difficulty":"numerical"},
    {"source":"For 2026, Australia has implemented a National Planning Level (NPL) cap of 295,000 international students.","answer":"The NPL cap for 2026 is set at 350,000 international student places across all sectors.","label":0,"difficulty":"numerical"},

    # Policy misstatement — plausible but wrong
    {"source":"Working Holiday Visa holders can work for the same employer for up to 6 months.","answer":"Working Holiday Visa holders can work for the same employer for up to 12 months without any restrictions.","label":0,"difficulty":"numerical"},
    {"source":"All student visa holders must maintain Overseas Student Health Cover (OSHC) for the entire duration of their stay.","answer":"Students must have OSHC for the first 12 months of their stay, after which they can switch to Medicare.","label":0,"difficulty":"semantic"},
    {"source":"Health examination results for visa applications are usually valid for 12 months.","answer":"Health examination results remain valid for 24 months from the date of examination.","label":0,"difficulty":"numerical"},
    {"source":"De facto relationships must have existed for at least 12 months before the visa application is lodged.","answer":"De facto partners must demonstrate 6 months of cohabitation before applying for the partner visa.","label":0,"difficulty":"numerical"},

    # Unsupported inference — adds plausible but fabricated detail
    {"source":"The Skilled Nominated Visa (Subclass 190) is a points-tested permanent visa. Nomination adds 5 points.","answer":"The Subclass 190 nomination adds 5 points, and applicants automatically receive an additional 10 bonus points for regional nominations.","label":0,"difficulty":"adversarial"},
    {"source":"New Zealand citizens can live and work in Australia on a Special Category Visa (Subclass 444).","answer":"NZ citizens on the Subclass 444 visa have full access to Medicare and all social security benefits from day one.","label":0,"difficulty":"adversarial"},
    {"source":"Financial evidence for a student visa must demonstrate access to at least AUD 29,710 per year for living costs.","answer":"Student visa applicants must show AUD 21,041 per year for living costs as per the updated 2026 requirements.","label":0,"difficulty":"numerical"},
    {"source":"The Prospective Marriage Visa (Subclass 300) allows the fiance to travel to Australia and marry within 9 months.","answer":"The Prospective Marriage Visa gives couples 12 months to marry after arrival in Australia.","label":0,"difficulty":"numerical"},
    {"source":"The general residence requirement for citizenship is four years, including at least 12 months on a permanent visa.","answer":"The residence requirement for citizenship is four years, including at least 24 months on a permanent visa.","label":0,"difficulty":"numerical"},
    {"source":"The Contributory Parent Visa (Subclass 143) has a second instalment of approximately AUD 47,755.","answer":"The second instalment for the Contributory Parent Visa is approximately AUD 43,600 as updated for 2026.","label":0,"difficulty":"numerical"},
]
# fmt: on

# Validate
from collections import Counter
grounded_v5 = sum(1 for d in hallucination_dataset_v5 if d['label'] == 1)
hallucinated_v5 = sum(1 for d in hallucination_dataset_v5 if d['label'] == 0)
diff_counts = Counter(d['difficulty'] for d in hallucination_dataset_v5)
print(f"v5 Dataset: {len(hallucination_dataset_v5)} samples ({grounded_v5} grounded, {hallucinated_v5} hallucinated)")
print(f"Difficulty distribution:")
for cat, n in diff_counts.most_common():
    print(f"  {cat}: {n}")


In [ ]:
# ════════════════════════════════════════════════════════════════
# U1b — Prepare v5 splits (same leakage-free method)
# ════════════════════════════════════════════════════════════════
train_data_v5, test_data_v5 = prepare_dataset(hallucination_dataset_v5, test_size=0.3, augment_multiplier=3, seed=42)

# Also create a validation set from training data for threshold tuning
from sklearn.model_selection import train_test_split as tts
train_final_v5, val_data_v5 = tts(train_data_v5, test_size=0.2, random_state=42,
                                   stratify=[d['label'] for d in train_data_v5])
print(f"Train: {len(train_final_v5)} | Val: {len(val_data_v5)} | Test: {len(test_data_v5)}")


### 4.5 Dataset Integrity Checks

Before interpreting model scores, the notebook checks for duplicated answers, exact cross-split leakage, and near-duplicate overlap. This makes the evaluation more credible and prevents accidental memorisation from being mistaken for semantic generalisation.


In [ ]:
# ============================================================
# E1 — Dataset split validation
# ============================================================
from difflib import SequenceMatcher

def validate_dataset_splits(train, test):
    """Check for duplicates and near-duplicates across splits."""
    print("DATASET INTEGRITY VALIDATION")
    print("=" * 60)

    # 1. Exact duplicates within each set
    train_answers = [d['answer'] for d in train]
    test_answers = [d['answer'] for d in test]

    train_dupes = len(train_answers) - len(set(train_answers))
    test_dupes = len(test_answers) - len(set(test_answers))
    print(f"Train exact duplicates (from augmentation): {train_dupes}")
    print(f"Test exact duplicates: {test_dupes}")

    # 2. Cross-split near-duplicates (similarity > 0.85)
    # Compare only original train answers (before augmentation) vs test
    train_originals = list(set(train_answers))  # deduplicate for speed
    near_dupes = 0
    high_sim_pairs = []

    for t_ans in test_answers:
        for tr_ans in train_originals[:100]:  # cap for speed
            sim = SequenceMatcher(None, t_ans.lower(), tr_ans.lower()).ratio()
            if sim > 0.85:
                near_dupes += 1
                if len(high_sim_pairs) < 3:
                    high_sim_pairs.append((sim, t_ans[:60], tr_ans[:60]))

    print(f"Cross-split near-duplicates (sim > 0.85): {near_dupes}")

    if high_sim_pairs:
        print("  Examples:")
        for sim, t, tr in high_sim_pairs:
            print(f"    sim={sim:.3f}")
            print(f"      test:  {t}...")
            print(f"      train: {tr}...")

    # 3. Label distribution
    train_labels = Counter(d['label'] for d in train)
    test_labels = Counter(d['label'] for d in test)
    print(f"\nLabel distribution:")
    print(f"  Train: {dict(train_labels)}")
    print(f"  Test:  {dict(test_labels)}")

    # 4. Embedding-based overlap check
    train_sample_embs = sbert_model.encode(test_answers[:20], convert_to_numpy=True)
    test_embs = sbert_model.encode(train_answers[:50], convert_to_numpy=True)

    # Compute max similarity of each test sample to any train sample
    max_sims = []
    for t_emb in train_sample_embs:
        sims = np.dot(test_embs, t_emb) / (
            np.linalg.norm(test_embs, axis=1) * np.linalg.norm(t_emb) + 1e-8
        )
        max_sims.append(np.max(sims))

    print(f"\nSemantic overlap (test vs train):")
    print(f"  Mean max similarity: {np.mean(max_sims):.4f}")
    print(f"  Max similarity:      {np.max(max_sims):.4f}")
    print(f"  Samples with sim > 0.95: {sum(1 for s in max_sims if s > 0.95)}")

    leakage_risk = "LOW" if np.max(max_sims) < 0.95 and near_dupes == 0 else "WARNING"
    print(f"\n  Leakage risk: {leakage_risk}")

    return {'train_dupes': train_dupes, 'test_dupes': test_dupes,
            'near_dupes': near_dupes, 'leakage_risk': leakage_risk}

split_validation = validate_dataset_splits(train_data, test_data)

**Interpretation note:** If leakage or near-duplicates are reported, detection results should be treated cautiously. If the split is clean, later performance is more likely to reflect genuine generalisation to unseen answer-source pairs.


## 5. Retrieval Module

Retrieval is the foundation of the system because verification is only meaningful when the answer is compared against relevant evidence. This section evaluates both dense semantic retrieval and hybrid retrieval.


### 5.1 Embedding Strategy

Word2Vec is included as a transparent classical embedding baseline, while Sentence-BERT is used for semantic retrieval. The comparison justifies why the final retriever uses contextual sentence embeddings rather than relying only on local word co-occurrence.


In [ ]:
# ============================================================
# 3.1 — Word2Vec (domain-specific)
# ============================================================
corpus_tokens = [word_tokenize(p['text'].lower()) for p in knowledge_base]
w2v_model = Word2Vec(sentences=corpus_tokens, vector_size=100, window=5,
                     min_count=2, workers=4, epochs=50, sg=1, seed=SEED)

print(f"Word2Vec vocab: {len(w2v_model.wv)}, dim: {w2v_model.wv.vector_size}")
for word in ['visa', 'student', 'australia', 'work', 'citizen']:
    if word in w2v_model.wv:
        top3 = w2v_model.wv.most_similar(word, topn=3)
        print(f"  '{word}' -> {[(w, f'{s:.2f}') for w, s in top3]}")

# ============================================================
# 3.2 — Load Sentence-BERT
# ============================================================
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# ============================================================
# 3.3 — Comparison on retrieval task
# ============================================================
def avg_vec(tokens, model, dim):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

test_q = "Can I work while studying in Australia?"
q_tok = word_tokenize(test_q.lower())

methods = {
    "Word2Vec": lambda t: avg_vec(word_tokenize(t.lower()), w2v_model, 100),
    "GloVe (spaCy)": lambda t: nlp_md(t).vector,
    "Sentence-BERT": lambda t: sbert_model.encode(t),
}

print(f"\nQuery: '{test_q}'")
for name, embed_fn in methods.items():
    q_emb = embed_fn(test_q)
    scores = []
    for i, p in enumerate(knowledge_base):
        d_emb = embed_fn(p['text'])
        sim = np.dot(q_emb, d_emb) / (np.linalg.norm(q_emb) * np.linalg.norm(d_emb) + 1e-8)
        scores.append((sim, i))
    scores.sort(reverse=True)
    top = scores[0]
    print(f"  {name:20s} -> [{knowledge_base[top[1]]['category']}] (sim={top[0]:.4f})")

print("\nConclusion: Sentence-BERT captures semantic meaning at sentence level.")
print("Selected for retrieval pipeline.")

### 5.2 Query Processing with POS and NER

POS tagging and NER are used to expose the linguistic structure of user questions. This is useful both for retrieval diagnostics and for explaining which entities or concepts the system attends to.


In [ ]:
def analyse_query(query):
    """Extract POS tags, entities, and key terms from a query."""
    doc = nlp_md(query)
    return {
        'nouns': [t.text for t in doc if t.pos_ in ('NOUN', 'PROPN')],
        'verbs': [t.text for t in doc if t.pos_ == 'VERB'],
        'entities': [(e.text, e.label_) for e in doc.ents],
        'noun_chunks': [c.text for c in doc.noun_chunks],
        'pos_sequence': ' '.join(f"{t.text}/{t.pos_}" for t in doc),
    }

# Demo
for q in [
    "How many hours can a student work per fortnight in Australia?",
    "What documents are needed for a partner visa application?",
    "Can New Zealand citizens get permanent residency in Australia?",
]:
    info = analyse_query(q)
    print(f"Q: '{q}'")
    print(f"  Nouns: {info['nouns']}, Entities: {info['entities']}")
    print(f"  POS: {info['pos_sequence']}")
    print()

### 5.3 Dense Retrieval and Cross-Encoder Re-ranking

FAISS performs fast dense retrieval over Sentence-BERT vectors. A cross-encoder then re-ranks candidate passages by jointly scoring the query and passage, which is slower but typically more precise.


In [ ]:
# ============================================================
# 5.1 — Build FAISS index
# ============================================================
kb_texts = [p['text'] for p in knowledge_base]
kb_embeddings = sbert_model.encode(kb_texts, convert_to_numpy=True, show_progress_bar=True)

dimension = kb_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
kb_norm = kb_embeddings.copy()
faiss.normalize_L2(kb_norm)
faiss_index.add(kb_norm)
print(f"FAISS index: {faiss_index.ntotal} passages, dim={dimension}")

# ============================================================
# 5.2 — Cross-encoder for re-ranking
# ============================================================
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)
print("Cross-encoder loaded for re-ranking.")

def retrieve_passages(query, top_k=5, rerank=True):
    """Retrieve passages with optional cross-encoder re-ranking."""
    # Stage 1: FAISS bi-encoder retrieval (get more candidates)
    q_emb = sbert_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    scores, indices = faiss_index.search(q_emb, min(top_k * 2, len(knowledge_base)))

    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        candidates.append({
            'bi_score': float(score),
            'passage': knowledge_base[idx],
            'text': knowledge_base[idx]['text'],
            'idx': int(idx),
        })

    # Stage 2: Cross-encoder re-ranking
    if rerank and candidates:
        pairs = [(query, c['text']) for c in candidates]
        ce_scores = cross_encoder.predict(pairs)
        for i, s in enumerate(ce_scores):
            candidates[i]['ce_score'] = float(s)
        candidates.sort(key=lambda x: x['ce_score'], reverse=True)
    else:
        for c in candidates:
            c['ce_score'] = c['bi_score']

    return candidates[:top_k]

# Quick test
r = retrieve_passages("How many hours can a student work?", top_k=3)
for i, p in enumerate(r):
    print(f"  {i+1}. [{p['passage']['category']}] bi={p['bi_score']:.3f} ce={p['ce_score']:.3f}")
    print(f"     {p['text'][:80]}...")

### 5.4 Retrieval Evaluation

Retrieval is evaluated with MRR and Recall@k. These metrics are important because a correct verifier cannot compensate for irrelevant retrieved evidence.


In [ ]:
# ============================================================
# Retrieval ground truth: query -> expected category
# ============================================================
retrieval_queries = [
    ("How many hours can a student work?", "Student Visa"),
    ("What is the visitor visa?", "Visitor Visa"),
    ("How do I become a citizen?", "Citizenship"),
    ("Can NZ citizens work in Australia?", "NZ Citizens"),
    ("What is a bridging visa?", "Bridging Visa"),
    ("How much does the partner visa cost?", "Partner Visa"),
    ("What documents do I need for a student visa?", "Student Visa"),
    ("What is the Working Holiday Visa?", "Work Visa"),
    ("Do I need health insurance as a student?", "Student Visa"),
    ("What happens if my visa is refused?", "Application Process"),
    ("Can visitors work in Australia?", "Visitor Visa"),
    ("What is the Global Talent Visa?", "Work Visa"),
    ("How long is the parent visa processing time?", "Family and Minors"),
    ("What is OSHC?", "Student Visa"),
    ("What is the citizenship test?", "Citizenship"),
    ("Can I travel on a Bridging Visa A?", "Bridging Visa"),
    ("What is the Skilled Independent Visa?", "Work Visa"),
    ("Do I need a police check for my visa?", "Health and Character"),
    ("What is a Protection Visa?", "Humanitarian"),
    ("What is VEVO?", "Visa Conditions"),
]

def evaluate_retrieval(queries, k_values=[1, 3, 5]):
    """Compute Recall@k, Top-1 accuracy, and MRR."""
    reciprocal_ranks = []
    recall = {k: 0 for k in k_values}

    for query, expected_cat in queries:
        results = retrieve_passages(query, top_k=max(k_values))
        categories = [r['passage']['category'] for r in results]

        # MRR
        rr = 0.0
        for rank, cat in enumerate(categories):
            if cat == expected_cat:
                rr = 1.0 / (rank + 1)
                break
        reciprocal_ranks.append(rr)

        # Recall@k
        for k in k_values:
            if expected_cat in categories[:k]:
                recall[k] += 1

    n = len(queries)
    mrr = np.mean(reciprocal_ranks)

    print("RETRIEVAL EVALUATION")
    print("=" * 50)
    print(f"MRR:        {mrr:.4f}")
    print(f"Top-1 Acc:  {recall[1]/n:.4f} ({recall[1]}/{n})")
    for k in k_values:
        print(f"Recall@{k}:   {recall[k]/n:.4f} ({recall[k]}/{n})")

    return {'mrr': mrr, 'recall': {k: recall[k]/n for k in k_values}}

ret_metrics = evaluate_retrieval(retrieval_queries)

In [ ]:
# ============================================================
# E2 — Retrieval before vs after re-ranking
# ============================================================
def evaluate_retrieval_comparison(queries, k_values=[1, 3, 5]):
    """Compare retrieval metrics with and without re-ranking."""

    results = {'sbert': {k: 0 for k in k_values},
               'reranked': {k: 0 for k in k_values}}
    mrr = {'sbert': [], 'reranked': []}

    for query, expected_cat in queries:
        # Without re-ranking
        no_rerank = retrieve_passages(query, top_k=max(k_values), rerank=False)
        cats_nr = [r['passage']['category'] for r in no_rerank]

        # With re-ranking
        with_rerank = retrieve_passages(query, top_k=max(k_values), rerank=True)
        cats_rr = [r['passage']['category'] for r in with_rerank]

        for method, cats, key in [('sbert', cats_nr, 'sbert'), ('reranked', cats_rr, 'reranked')]:
            rr = 0.0
            for rank, cat in enumerate(cats):
                if cat == expected_cat:
                    rr = 1.0 / (rank + 1)
                    break
            mrr[key].append(rr)

            for k in k_values:
                if expected_cat in cats[:k]:
                    results[key][k] += 1

    n = len(queries)
    print("RETRIEVAL: SBERT vs CROSS-ENCODER RE-RANKED")
    print("=" * 60)
    print(f"{'Metric':<20} {'SBERT Only':>15} {'+ Re-ranking':>15} {'Delta':>10}")
    print("-" * 60)

    mrr_s = np.mean(mrr['sbert'])
    mrr_r = np.mean(mrr['reranked'])
    print(f"{'MRR':<20} {mrr_s:>15.4f} {mrr_r:>15.4f} {mrr_r-mrr_s:>+10.4f}")

    for k in k_values:
        rs = results['sbert'][k] / n
        rr = results['reranked'][k] / n
        print(f"{'Recall@'+str(k):<20} {rs:>15.4f} {rr:>15.4f} {rr-rs:>+10.4f}")

    # Category-level accuracy (top-1)
    print(f"\n{'Category Accuracy':<20}")
    cat_correct = {'sbert': Counter(), 'reranked': Counter()}
    cat_total = Counter()
    for query, expected_cat in queries:
        cat_total[expected_cat] += 1
        nr = retrieve_passages(query, top_k=1, rerank=False)
        rr = retrieve_passages(query, top_k=1, rerank=True)
        if nr[0]['passage']['category'] == expected_cat:
            cat_correct['sbert'][expected_cat] += 1
        if rr[0]['passage']['category'] == expected_cat:
            cat_correct['reranked'][expected_cat] += 1

    for cat in sorted(cat_total.keys()):
        s_acc = cat_correct['sbert'][cat] / cat_total[cat]
        r_acc = cat_correct['reranked'][cat] / cat_total[cat]
        print(f"  {cat:<25} SBERT={s_acc:.2f}  Reranked={r_acc:.2f}")

    return {'mrr_sbert': mrr_s, 'mrr_reranked': mrr_r}

ret_comparison = evaluate_retrieval_comparison(retrieval_queries)

**Retrieval interpretation:** Prefer improvements in MRR and Recall@1 because they indicate that the most relevant evidence is being surfaced early. Recall@5 remains useful because generation can condition on multiple retrieved passages.


### 5.5 Hybrid Retrieval: BM25 + Sentence-BERT

Hybrid retrieval combines lexical matching with semantic similarity. BM25 helps preserve exact terms such as visa subclass numbers, while Sentence-BERT captures paraphrases and semantically related queries.


In [ ]:
# ════════════════════════════════════════════════════════════════
# U4 — HYBRID RETRIEVAL (BM25 + SBERT)
# ════════════════════════════════════════════════════════════════

# BM25 implementation (lightweight, no external dependency)
import math

class BM25:
    """Okapi BM25 implementation for document retrieval."""
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.corpus = corpus
        self.doc_len = []
        self.df = {}  # document frequency
        self.idf = {}
        self.doc_terms = []
        self.avgdl = 0

        # Build index
        for doc in corpus:
            terms = doc.lower().split()
            self.doc_terms.append(Counter(terms))
            self.doc_len.append(len(terms))
            for term in set(terms):
                self.df[term] = self.df.get(term, 0) + 1

        self.avgdl = sum(self.doc_len) / len(self.doc_len)
        N = len(corpus)
        for term, df in self.df.items():
            self.idf[term] = math.log((N - df + 0.5) / (df + 0.5) + 1)

    def score(self, query, doc_idx):
        q_terms = query.lower().split()
        doc_terms = self.doc_terms[doc_idx]
        dl = self.doc_len[doc_idx]
        score = 0.0
        for term in q_terms:
            if term not in doc_terms:
                continue
            tf = doc_terms[term]
            idf = self.idf.get(term, 0)
            numerator = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            score += idf * numerator / denominator
        return score

    def rank(self, query, top_k=10):
        scores = [(i, self.score(query, i)) for i in range(len(self.corpus))]
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

# Build BM25 index
bm25_index = BM25(kb_texts)
print(f"BM25 index built: {len(kb_texts)} documents")


def retrieve_hybrid(query, top_k=5, alpha=0.5, rerank=True):
    """Hybrid retrieval combining SBERT and BM25 scores.
    alpha: weight for SBERT (1-alpha for BM25)
    """
    # SBERT scores
    q_emb = sbert_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    sbert_scores_raw, sbert_indices = faiss_index.search(q_emb, len(knowledge_base))

    # Normalise SBERT scores to [0, 1]
    sbert_min = sbert_scores_raw[0].min()
    sbert_max = sbert_scores_raw[0].max()
    sbert_range = sbert_max - sbert_min if sbert_max > sbert_min else 1.0

    sbert_norm = {}
    for score, idx in zip(sbert_scores_raw[0], sbert_indices[0]):
        sbert_norm[int(idx)] = (float(score) - sbert_min) / sbert_range

    # BM25 scores
    bm25_results = bm25_index.rank(query, top_k=len(knowledge_base))
    bm25_max = max(s for _, s in bm25_results) if bm25_results else 1.0
    bm25_max = bm25_max if bm25_max > 0 else 1.0

    bm25_norm = {}
    for idx, score in bm25_results:
        bm25_norm[idx] = score / bm25_max

    # Combine
    all_indices = set(sbert_norm.keys()) | set(bm25_norm.keys())
    combined = []
    for idx in all_indices:
        s_score = sbert_norm.get(idx, 0.0)
        b_score = bm25_norm.get(idx, 0.0)
        hybrid_score = alpha * s_score + (1 - alpha) * b_score
        combined.append({
            'idx': idx,
            'sbert_score': s_score,
            'bm25_score': b_score,
            'hybrid_score': hybrid_score,
            'passage': knowledge_base[idx],
            'text': knowledge_base[idx]['text'],
        })

    combined.sort(key=lambda x: x['hybrid_score'], reverse=True)
    candidates = combined[:top_k * 2]

    # Optional cross-encoder re-ranking
    if rerank and candidates:
        pairs = [(query, c['text']) for c in candidates]
        ce_scores = cross_encoder.predict(pairs)
        for i, s in enumerate(ce_scores):
            candidates[i]['ce_score'] = float(s)
        candidates.sort(key=lambda x: x['ce_score'], reverse=True)
    else:
        for c in candidates:
            c['ce_score'] = c['hybrid_score']

    return candidates[:top_k]


def evaluate_retrieval_methods(queries, k_values=[1, 3, 5]):
    """Compare SBERT-only, BM25-only, and Hybrid retrieval."""
    methods = {
        'SBERT Only': lambda q, k: retrieve_passages(q, top_k=k, rerank=False),
        'BM25 Only': lambda q, k: _bm25_retrieve(q, k),
        'Hybrid (α=0.5)': lambda q, k: retrieve_hybrid(q, top_k=k, alpha=0.5, rerank=False),
        'Hybrid + Rerank': lambda q, k: retrieve_hybrid(q, top_k=k, alpha=0.5, rerank=True),
    }

    print("RETRIEVAL METHOD COMPARISON")
    print("=" * 70)
    print(f"{'Method':<25} {'MRR':>8} {'R@1':>8} {'R@3':>8} {'R@5':>8}")
    print("-" * 70)

    all_results = {}
    for name, retrieve_fn in methods.items():
        reciprocal_ranks = []
        recall = {k: 0 for k in k_values}

        for query, expected_cat in queries:
            results = retrieve_fn(query, max(k_values))
            categories = [r['passage']['category'] for r in results]

            rr = 0.0
            for rank, cat in enumerate(categories):
                if cat == expected_cat:
                    rr = 1.0 / (rank + 1)
                    break
            reciprocal_ranks.append(rr)

            for k in k_values:
                if expected_cat in categories[:k]:
                    recall[k] += 1

        n = len(queries)
        mrr = np.mean(reciprocal_ranks)
        r_at = {k: recall[k] / n for k in k_values}
        print(f"{name:<25} {mrr:>8.4f} {r_at[1]:>8.4f} {r_at[3]:>8.4f} {r_at[5]:>8.4f}")
        all_results[name] = {'mrr': mrr, 'recall': r_at, 'rrs': reciprocal_ranks}

    return all_results

def _bm25_retrieve(query, top_k):
    """BM25-only retrieval (wrapper for comparison)."""
    results = bm25_index.rank(query, top_k)
    return [{'passage': knowledge_base[idx], 'text': knowledge_base[idx]['text'],
             'bm25_score': score, 'ce_score': score} for idx, score in results]

retrieval_comparison = evaluate_retrieval_methods(retrieval_queries)


**Hybrid retrieval interpretation:** If the hybrid method improves MRR or Recall@1, it suggests that exact visa terms and semantic meaning are complementary. If gains are small, the analysis still provides useful evidence that dense retrieval is already strong for this curated domain.


## 6. Answer Generation

The generation module converts retrieved evidence into a natural-language answer. It is intentionally lightweight so the experiment remains runnable in free Colab.

The generator is not treated as the main contribution. Its role is to create an end-to-end QA setting in which hallucination verification can be tested.


In [ ]:
# ============================================================
# Load Flan-T5-base (upgrade from flan-t5-small)
# ============================================================
gen_tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-base')
gen_model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base').to(device)
print(f"Generator: flan-t5-base ({sum(p.numel() for p in gen_model.parameters()):,} params)")

def generate_answer(query, retrieved_passages, max_length=256):
    """Generate an answer conditioned on retrieved context."""
    context = " ".join([p['text'] for p in retrieved_passages[:3]])

    prompt = f"""You are an Australian visa information assistant. Answer the question
using ONLY the provided context. If the context does not contain enough
information to answer fully, say "Based on available information" and provide
what you can. Do not make up facts.

Context: {context}

Question: {query}

Answer:"""

    inputs = gen_tokenizer(prompt, return_tensors="pt", max_length=512,
                           truncation=True).to(device)
    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs, max_length=max_length, num_beams=4,
            early_stopping=True, no_repeat_ngram_size=3,
            length_penalty=1.2,  # encourage longer answers
        )
    return gen_tokenizer.decode(outputs[0], skip_special_tokens=True), context

# ============================================================
# Generation evaluation
# ============================================================
def evaluate_generation(queries):
    """Evaluate generation quality: failure rate, avg length."""
    failures = 0
    lengths = []
    results = []

    for query, _ in queries:
        retrieved = retrieve_passages(query, top_k=5)
        answer, ctx = generate_answer(query, retrieved)
        word_count = len(answer.split())
        lengths.append(word_count)

        is_fail = (word_count < 3 or
                   answer.lower().strip() in ['unanswerable', 'yes', 'no', 'yes.', 'no.', ''] or
                   answer.strip() == '')
        if is_fail:
            failures += 1

        results.append({'query': query, 'answer': answer, 'words': word_count, 'fail': is_fail})

    n = len(queries)
    print("\nGENERATION EVALUATION")
    print("=" * 50)
    print(f"Total queries:   {n}")
    print(f"Failure rate:    {failures/n:.2%} ({failures}/{n})")
    print(f"Avg answer len:  {np.mean(lengths):.1f} words")
    print(f"Min/Max length:  {min(lengths)}/{max(lengths)} words")
    print("\nSample outputs:")
    for r in results[:5]:
        status = "FAIL" if r['fail'] else "OK"
        print(f"  [{status}] Q: {r['query']}")
        print(f"         A: {r['answer'][:100]}...")
    return results

gen_results = evaluate_generation(retrieval_queries[:10])

**Generation interpretation:** Short or generic answers should be interpreted as a limitation of the lightweight generator, not as a failure of the verification idea. The strongest claim of the project is about checking groundedness, not producing the highest-quality answer text.


## 7. Hallucination Verification Pipeline

The verifier combines three complementary signals:

- **NER and number matching:** checks whether entities and numerical facts in the answer appear in the source.
- **NLI entailment:** estimates whether the source entails, contradicts, or is neutral toward the answer.
- **Embedding similarity:** measures broad semantic relatedness between answer and evidence.

The fused score is intentionally interpretable: each signal corresponds to a different hallucination failure mode.


In [ ]:
# ============================================================
# Layer A: NER cross-checking
# ============================================================
def extract_entities(text):
    doc = nlp_sm(text)
    return set((e.text.lower().strip(), e.label_) for e in doc.ents)

def extract_numbers(text):
    return set(re.findall(r'\b\d+(?:,\d+)*(?:\.\d+)?\b', text))

def ner_score(source, answer):
    """Score: 1.0 = all entities grounded, 0.0 = none grounded."""
    s_ent, a_ent = extract_entities(source), extract_entities(answer)
    s_num, a_num = extract_numbers(source), extract_numbers(answer)
    unsup = len(a_ent - s_ent) + len(a_num - s_num)
    total = len(a_ent) + len(a_num)
    if total == 0:
        return 1.0
    return max(0.0, 1.0 - unsup / total)

# ============================================================
# Layer B: NLI entailment
# ============================================================
nli_tokenizer = AutoTokenizer.from_pretrained("cross-encoder/nli-deberta-v3-small")
nli_model = AutoModelForSequenceClassification.from_pretrained(
    "cross-encoder/nli-deberta-v3-small").to(device)
NLI_LABELS = {0: "contradiction", 1: "entailment", 2: "neutral"}

def nli_score(source, answer):
    """Returns (entailment_probability, label_string)."""
    inputs = nli_tokenizer(source, answer, return_tensors="pt",
                           truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        probs = F.softmax(nli_model(**inputs).logits, dim=1)[0]
    label = NLI_LABELS[torch.argmax(probs).item()]
    return probs[1].item(), label

# ============================================================
# Layer C: Embedding similarity
# ============================================================
def embedding_score(source, answer):
    embs = sbert_model.encode([source, answer], convert_to_numpy=True)
    return float(np.dot(embs[0], embs[1]) /
                 (np.linalg.norm(embs[0]) * np.linalg.norm(embs[1])))

# ============================================================
# Combined pipeline
# ============================================================
def detect_hallucination(source, answer, weights=(0.2, 0.5, 0.3), threshold=0.6):
    """Multi-layer hallucination detection. Returns dict with scores and verdict."""
    w_ner, w_nli, w_emb = weights
    s_ner = ner_score(source, answer)
    s_nli, nli_label = nli_score(source, answer)
    s_emb = embedding_score(source, answer)
    combined = w_ner * s_ner + w_nli * s_nli + w_emb * s_emb

    if combined >= threshold:
        verdict = "GROUNDED"
    elif combined >= 0.35:
        verdict = "PARTIALLY GROUNDED"
    else:
        verdict = "HALLUCINATED"

    return {
        'ner': s_ner, 'nli': s_nli, 'nli_label': nli_label,
        'emb': s_emb, 'combined': combined, 'verdict': verdict
    }

# Quick test
r = detect_hallucination(
    "Student visa holders can work up to 48 hours per fortnight.",
    "Students can work unlimited hours with no restrictions."
)
print(f"Test: NER={r['ner']:.2f} NLI={r['nli']:.2f} EMB={r['emb']:.2f} -> {r['verdict']}")

### 7.1 Threshold Tuning

The threshold controls when the verifier labels an answer as grounded. Optimising the threshold on validation data avoids choosing a decision boundary purely by intuition.


In [ ]:
# ════════════════════════════════════════════════════════════════
# U3 — THRESHOLD OPTIMISATION
# ════════════════════════════════════════════════════════════════

def optimise_threshold(val_data, weights=(0.2, 0.5, 0.3)):
    """Find optimal threshold by maximising F1 on validation set."""
    # Compute combined scores for all val samples
    scores_and_labels = []
    for sample in val_data:
        result = detect_hallucination(sample['source'], sample['answer'], weights, threshold=0.5)
        scores_and_labels.append((result['combined'], sample['label']))

    scores_and_labels.sort(key=lambda x: x[0])
    all_scores = [s for s, _ in scores_and_labels]

    # Sweep thresholds
    thresholds = np.arange(0.1, 0.95, 0.01)
    results = []
    for t in thresholds:
        preds = [1 if s >= t else 0 for s, _ in scores_and_labels]
        trues = [l for _, l in scores_and_labels]
        f1 = f1_score(trues, preds, average='weighted')
        prec = precision_score(trues, preds, average='weighted', zero_division=0)
        rec = recall_score(trues, preds, average='weighted', zero_division=0)
        results.append({'threshold': t, 'f1': f1, 'precision': prec, 'recall': rec})

    # Find best
    best = max(results, key=lambda x: x['f1'])

    print("THRESHOLD OPTIMISATION")
    print("=" * 60)
    print(f"Validation set size: {len(val_data)}")
    print(f"Optimal threshold:   {best['threshold']:.2f}")
    print(f"Best F1:             {best['f1']:.4f}")
    print(f"Precision at best:   {best['precision']:.4f}")
    print(f"Recall at best:      {best['recall']:.4f}")

    return best, results

threshold_best, threshold_curve = optimise_threshold(val_data_v5)
OPTIMAL_THRESHOLD = threshold_best['threshold']


**Threshold interpretation:** The selected threshold should be reported as validation-tuned rather than universally optimal. On a larger or different dataset, the threshold may need recalibration.


### 7.2 Category-Wise Verification Performance

Category-wise analysis shows whether the verifier handles different hallucination types consistently. This is especially important for hard negatives, where answers may be lexically similar but factually wrong.


In [ ]:
# ════════════════════════════════════════════════════════════════
# U2 — CATEGORY-WISE ACCURACY BREAKDOWN
# ════════════════════════════════════════════════════════════════

def evaluate_by_category(dataset, weights=(0.2, 0.5, 0.3), threshold=0.6):
    """Evaluate detection accuracy per difficulty category."""
    from collections import defaultdict
    cat_results = defaultdict(lambda: {'correct': 0, 'total': 0, 'samples': []})

    for sample in dataset:
        cat = sample.get('difficulty', 'unknown')
        result = detect_hallucination(sample['source'], sample['answer'], weights, threshold)
        pred = 1 if result['verdict'] == 'GROUNDED' else 0
        correct = (pred == sample['label'])
        cat_results[cat]['total'] += 1
        cat_results[cat]['correct'] += correct
        if not correct:
            cat_results[cat]['samples'].append({
                'answer': sample['answer'][:80],
                'true': sample['label'],
                'pred': pred,
                'scores': result,
            })

    print("CATEGORY-WISE ACCURACY")
    print("=" * 60)
    print(f"{'Category':<25} {'Accuracy':>10} {'Correct':>10} {'Total':>8}")
    print("-" * 60)
    for cat in sorted(cat_results.keys()):
        r = cat_results[cat]
        acc = r['correct'] / r['total'] if r['total'] > 0 else 0
        print(f"{cat:<25} {acc:>10.4f} {r['correct']:>10} {r['total']:>8}")

    overall = sum(r['correct'] for r in cat_results.values())
    total = sum(r['total'] for r in cat_results.values())
    print("-" * 60)
    print(f"{'OVERALL':<25} {overall/total:>10.4f} {overall:>10} {total:>8}")

    # Show example failures per category
    print("\nSample failures by category:")
    for cat in sorted(cat_results.keys()):
        fails = cat_results[cat]['samples']
        if fails:
            print(f"\n  [{cat}] ({len(fails)} failures)")
            for f in fails[:2]:
                true_label = "GROUNDED" if f['true'] == 1 else "HALLUC"
                pred_label = "GROUNDED" if f['pred'] == 1 else "HALLUC"
                print(f"    True={true_label} Pred={pred_label}: {f['answer']}...")

    return cat_results

cat_results = evaluate_by_category(test_data_v5)


**Category interpretation:** Numerical and entity-confusion failures are particularly important in the visa domain because small factual changes can materially alter advice.


### 7.3 Explainable Verification Output

The explainability layer is not a separate model. It turns the verifier's intermediate scores into a structured explanation so users can see whether the decision was driven by unsupported numbers, unsupported entities, NLI contradiction, or low semantic similarity.


In [ ]:
# ════════════════════════════════════════════════════════════════
# U10 — EXPLAINABILITY / STRUCTURED REASONING OUTPUT
# ════════════════════════════════════════════════════════════════

def detect_hallucination_explained(source, answer, retrieved_passages=None,
                                    weights=(0.2, 0.5, 0.3), threshold=None):
    """Enhanced hallucination detection with structured reasoning output."""
    if threshold is None:
        threshold = OPTIMAL_THRESHOLD

    w_ner, w_nli, w_emb = weights
    s_ner = ner_score(source, answer)
    s_nli, nli_label = nli_score(source, answer)
    s_emb = embedding_score(source, answer)
    combined = w_ner * s_ner + w_nli * s_nli + w_emb * s_emb

    if combined >= threshold:
        verdict = "GROUNDED"
    elif combined >= 0.35:
        verdict = "PARTIALLY GROUNDED"
    else:
        verdict = "HALLUCINATED"

    # Build reasoning chain
    reasons = []

    # NER analysis
    src_ents = extract_entities(source)
    ans_ents = extract_entities(answer)
    unsup_ents = ans_ents - src_ents
    src_nums = extract_numbers(source)
    ans_nums = extract_numbers(answer)
    unsup_nums = ans_nums - src_nums

    if unsup_nums:
        reasons.append(f"Numerical inconsistency: answer contains {unsup_nums} not found in source")
    if unsup_ents:
        ent_strs = [f"{e[0]} ({e[1]})" for e in list(unsup_ents)[:3]]
        reasons.append(f"Unsupported entities: {', '.join(ent_strs)}")
    if s_ner == 1.0:
        reasons.append("All entities and numbers are grounded in source")

    # NLI analysis
    if nli_label == 'contradiction':
        reasons.append(f"NLI detects CONTRADICTION (confidence: {s_nli:.3f})")
    elif nli_label == 'entailment':
        reasons.append(f"NLI confirms ENTAILMENT (confidence: {s_nli:.3f})")
    else:
        reasons.append(f"NLI returns NEUTRAL — answer may extend beyond source (confidence: {s_nli:.3f})")

    # Embedding analysis
    if s_emb > 0.85:
        reasons.append(f"High semantic similarity ({s_emb:.3f}) — answer closely mirrors source")
    elif s_emb > 0.6:
        reasons.append(f"Moderate semantic similarity ({s_emb:.3f}) — answer is related but divergent")
    else:
        reasons.append(f"Low semantic similarity ({s_emb:.3f}) — answer appears substantially different from source")

    # Retrieval context
    if retrieved_passages:
        top_cat = retrieved_passages[0]['passage']['category']
        top_score = retrieved_passages[0].get('ce_score', retrieved_passages[0].get('bi_score', 0))
        reasons.append(f"Top retrieval: [{top_cat}] (score: {top_score:.3f})")

    return {
        'verdict': verdict,
        'combined': combined,
        'threshold': threshold,
        'ner': s_ner, 'nli': s_nli, 'nli_label': nli_label, 'emb': s_emb,
        'reasons': reasons,
    }


def demo_explained(query):
    """Full pipeline demonstration with explainability."""
    print(f"\n{'='*70}")
    print(f"QUERY: {query}")
    print(f"{'='*70}")

    info = analyse_query(query)
    print(f"Key terms: {info['nouns']} | Entities: {info['entities']}")

    retrieved = retrieve_hybrid(query, top_k=5, alpha=0.5, rerank=True)
    print(f"Retrieved: [{retrieved[0]['passage']['category']}] (ce={retrieved[0].get('ce_score', 0):.3f})")

    answer, _ = generate_answer(query, retrieved)
    print(f"Answer: {answer}")

    result = detect_hallucination_explained(
        retrieved[0]['text'], answer, retrieved_passages=retrieved
    )

    emoji = {"GROUNDED": "✅", "PARTIALLY GROUNDED": "⚠️", "HALLUCINATED": "❌"}
    print(f"\n{emoji.get(result['verdict'], '')} Verdict: {result['verdict']}")
    print(f"   Combined score: {result['combined']:.3f} (threshold: {result['threshold']:.2f})")
    print(f"   NER={result['ner']:.3f}  NLI={result['nli']:.3f} ({result['nli_label']})  EMB={result['emb']:.3f}")
    print(f"\n   Reasoning:")
    for i, reason in enumerate(result['reasons'], 1):
        print(f"   {i}. {reason}")

# Run demos
for q in [
    "Can I work while studying in Australia on a student visa?",
    "How do I become an Australian citizen?",
    "How much does the partner visa cost?",
    "Can NZ citizens live permanently in Australia?",
    "What health insurance do I need as a student?",
]:
    demo_explained(q)


## 8. Baseline Models

Baselines establish whether the proposed semantic verifier is necessary. The notebook compares the verifier against shallow neural models and simpler similarity-based methods.

The expected pattern is that Text CNNs and BiLSTMs may learn surface patterns but struggle with semantic entailment and subtle hard negatives.


In [ ]:
# ============================================================
# Model definitions
# ============================================================
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_filters=64,
                 filter_sizes=[2, 3, 4], pad_idx=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, fs) for fs in filter_sizes
        ])
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(num_filters * len(filter_sizes), 2)

    def forward(self, x):
        x = self.emb(x).permute(0, 2, 1)
        pooled = [F.relu(c(x)).max(dim=2)[0] for c in self.convs]
        return self.fc(self.drop(torch.cat(pooled, dim=1)))

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden=64, pad_idx=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden, batch_first=True,
                            bidirectional=True, num_layers=2, dropout=0.3)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden * 2, 2)

    def forward(self, x):
        x = self.emb(x)
        _, (h, _) = self.lstm(x)
        h = torch.cat((h[-2], h[-1]), dim=1)
        return self.fc(self.drop(h))

# ============================================================
# Tokenizer + Dataset
# ============================================================
class Tokenizer:
    def __init__(self):
        self.w2i = {"<pad>": 0, "<unk>": 1}
        self.n = 2

    def fit(self, texts):
        for t in texts:
            for w in t.lower().split():
                if w not in self.w2i:
                    self.w2i[w] = self.n
                    self.n += 1

    def encode(self, text, max_len=256):
        ids = [self.w2i.get(w, 1) for w in text.lower().split()]
        return (ids + [0] * max(0, max_len - len(ids)))[:max_len]

    @property
    def vocab_size(self):
        return len(self.w2i)

class PairDataset(Dataset):
    def __init__(self, data, tok, max_len=256):
        self.data = data
        self.tok = tok
        self.ml = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        d = self.data[i]
        ids = self.tok.encode(f"{d['source']} [SEP] {d['answer']}", self.ml)
        return torch.tensor(ids, dtype=torch.long), torch.tensor(d['label'], dtype=torch.long)

# ============================================================
# Build dataloaders
# ============================================================
tok = Tokenizer()
all_texts = [f"{d['source']} [SEP] {d['answer']}" for d in train_data + test_data]
tok.fit(all_texts)

train_ds = PairDataset(train_data, tok)
test_ds = PairDataset(test_data, tok)
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=8, shuffle=False)

print(f"Vocab: {tok.vocab_size} | Train: {len(train_data)} | Test: {len(test_data)}")

In [ ]:
def train(model, loader, epochs=50, lr=0.001):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    model.train()
    losses = []
    for ep in range(epochs):
        ep_loss = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
            ep_loss += loss.item()
        losses.append(ep_loss / len(loader))
        if (ep + 1) % 10 == 0:
            print(f"  Epoch {ep+1}/{epochs} loss={losses[-1]:.4f}")
    return losses

def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            p = torch.argmax(model(x.to(device)), dim=1).cpu().numpy()
            preds.extend(p)
            labels.extend(y.numpy())
    return preds, labels

# Train CNN
print("Training Text CNN...")
cnn = TextCNN(tok.vocab_size).to(device)
cnn_losses = train(cnn, train_dl)
cnn_pred, cnn_true = evaluate(cnn, test_dl)
print("\nText CNN Results:")
print(classification_report(cnn_true, cnn_pred, target_names=['Hallucinated', 'Grounded']))

# Train BiLSTM
print("Training BiLSTM...")
lstm = BiLSTM(tok.vocab_size).to(device)
lstm_losses = train(lstm, train_dl)
lstm_pred, lstm_true = evaluate(lstm, test_dl)
print("\nBiLSTM Results:")
print(classification_report(lstm_true, lstm_pred, target_names=['Hallucinated', 'Grounded']))

### 8.1 Overfitting Diagnostics

The train-test gap is reported to distinguish genuine generalisation from memorisation. This is particularly important because the labelled dataset is modest in size.


In [ ]:
# ============================================================
# E4 — Overfitting check for CNN and BiLSTM
# ============================================================
def overfitting_check(model, train_loader, test_loader, name):
    """Compare train vs test accuracy to detect overfitting."""
    model.eval()

    # Train accuracy
    train_preds, train_labels = [], []
    with torch.no_grad():
        for x, y in train_loader:
            p = torch.argmax(model(x.to(device)), dim=1).cpu().numpy()
            train_preds.extend(p)
            train_labels.extend(y.numpy())
    train_acc = accuracy_score(train_labels, train_preds)

    # Test accuracy
    test_preds, test_labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            p = torch.argmax(model(x.to(device)), dim=1).cpu().numpy()
            test_preds.extend(p)
            test_labels.extend(y.numpy())
    test_acc = accuracy_score(test_labels, test_preds)

    gap = train_acc - test_acc
    flag = "OVERFITTING" if gap > 0.15 else "ACCEPTABLE" if gap > 0.05 else "OK"

    print(f"  {name:<15} Train={train_acc:.4f}  Test={test_acc:.4f}  "
          f"Gap={gap:+.4f}  [{flag}]")

    return {'train': train_acc, 'test': test_acc, 'gap': gap, 'flag': flag}

print("OVERFITTING DIAGNOSTIC")
print("=" * 60)
cnn_overfit = overfitting_check(cnn, train_dl, test_dl, "Text CNN")
lstm_overfit = overfitting_check(lstm, train_dl, test_dl, "BiLSTM")

print(f"\nInterpretation:")
if cnn_overfit['flag'] == 'OVERFITTING':
    print(f"  Text CNN: train-test gap of {cnn_overfit['gap']:.2%} indicates overfitting.")
    print(f"  This is expected — CNNs memorise surface patterns from augmented data")
    print(f"  but cannot generalise semantic entailment to unseen pairs.")
if lstm_overfit['flag'] == 'OVERFITTING':
    print(f"  BiLSTM: train-test gap of {lstm_overfit['gap']:.2%} indicates overfitting.")
    print(f"  Sequential models need more diverse training data to generalise.")

**Overfitting interpretation:** A large train-test gap supports the argument that shallow supervised baselines are brittle on small curated hallucination datasets.


### 8.2 Stronger Non-Generative Baselines

The v5 baseline suite adds TF-IDF similarity, embedding-only similarity, NLI-only detection, and NER-only checking. These baselines clarify which components contribute most to the final verifier.


In [ ]:
# ════════════════════════════════════════════════════════════════
# U7 — STRONGER BASELINES
# ════════════════════════════════════════════════════════════════

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

def evaluate_baselines(dataset):
    """Compare simple baselines against the full pipeline."""
    results = {}

    # --- Baseline 1: TF-IDF similarity ---
    sources = [d['source'] for d in dataset]
    answers = [d['answer'] for d in dataset]
    trues = [d['label'] for d in dataset]

    tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
    all_texts = sources + answers
    tfidf.fit(all_texts)

    src_vecs = tfidf.transform(sources)
    ans_vecs = tfidf.transform(answers)

    tfidf_sims = []
    for i in range(len(dataset)):
        sim = sk_cosine(src_vecs[i], ans_vecs[i])[0, 0]
        tfidf_sims.append(sim)

    # Find best threshold for TF-IDF
    best_tfidf_f1, best_tfidf_t = 0, 0.5
    for t in np.arange(0.1, 0.9, 0.01):
        preds = [1 if s >= t else 0 for s in tfidf_sims]
        f1 = f1_score(trues, preds, average='weighted')
        if f1 > best_tfidf_f1:
            best_tfidf_f1 = f1
            best_tfidf_t = t
    results['TF-IDF Similarity'] = best_tfidf_f1

    # --- Baseline 2: Embedding similarity only ---
    emb_preds = []
    for d in dataset:
        embs = sbert_model.encode([d['source'], d['answer']], convert_to_numpy=True)
        sim = float(np.dot(embs[0], embs[1]) / (np.linalg.norm(embs[0]) * np.linalg.norm(embs[1])))
        emb_preds.append(sim)

    best_emb_f1, best_emb_t = 0, 0.5
    for t in np.arange(0.3, 0.95, 0.01):
        preds = [1 if s >= t else 0 for s in emb_preds]
        f1 = f1_score(trues, preds, average='weighted')
        if f1 > best_emb_f1:
            best_emb_f1 = f1
            best_emb_t = t
    results['Embedding Similarity'] = best_emb_f1

    # --- Baseline 3: NLI only ---
    nli_preds = []
    for d in dataset:
        _, label = nli_score(d['source'], d['answer'])
        nli_preds.append(1 if label == 'entailment' else 0)
    results['NLI Only (DeBERTa)'] = f1_score(trues, nli_preds, average='weighted')

    # --- Baseline 4: NER only ---
    best_ner_f1 = 0
    ner_scores_list = [ner_score(d['source'], d['answer']) for d in dataset]
    for t in np.arange(0.3, 1.0, 0.05):
        preds = [1 if s >= t else 0 for s in ner_scores_list]
        f1 = f1_score(trues, preds, average='weighted')
        if f1 > best_ner_f1:
            best_ner_f1 = f1
    results['NER Cross-check'] = best_ner_f1

    # --- Full pipeline ---
    pipe_preds = []
    for d in dataset:
        r = detect_hallucination(d['source'], d['answer'], threshold=OPTIMAL_THRESHOLD)
        pipe_preds.append(1 if r['verdict'] == 'GROUNDED' else 0)
    results['Full Pipeline'] = f1_score(trues, pipe_preds, average='weighted')

    # --- Text CNN and BiLSTM (from v4) ---
    # Re-evaluate on v5 test set
    tok_v5 = Tokenizer()
    all_texts_v5 = [f"{d['source']} [SEP] {d['answer']}" for d in test_data_v5]
    tok_v5.fit(all_texts_v5)
    test_ds_v5 = PairDataset(test_data_v5, tok_v5)
    test_dl_v5 = DataLoader(test_ds_v5, batch_size=8, shuffle=False)

    try:
        cnn_pred_v5, cnn_true_v5 = evaluate(cnn, test_dl_v5)
        results['Text CNN'] = f1_score(cnn_true_v5, cnn_pred_v5, average='weighted')
    except:
        results['Text CNN'] = 0.0

    try:
        lstm_pred_v5, lstm_true_v5 = evaluate(lstm, test_dl_v5)
        results['BiLSTM'] = f1_score(lstm_true_v5, lstm_pred_v5, average='weighted')
    except:
        results['BiLSTM'] = 0.0

    print("BASELINE COMPARISON")
    print("=" * 60)
    print(f"{'Method':<30} {'F1 (weighted)':>15}")
    print("-" * 50)
    for name, f1 in sorted(results.items(), key=lambda x: -x[1]):
        marker = " <<<" if name == 'Full Pipeline' else ""
        print(f"{name:<30} {f1:>15.4f}{marker}")

    return results

baseline_results = evaluate_baselines(test_data_v5)


**Baseline interpretation:** If NLI is the strongest individual component, the project should emphasise semantic entailment as the core detection mechanism. The full pipeline remains valuable when it improves robustness, interpretability, or category coverage.


## 9. Evaluation Framework

The evaluation combines standard classification metrics with retrieval metrics, ablation, external sanity tests, calibration, and statistical checks. The goal is not to maximise the number of metrics, but to make the central claim testable from several angles.


### 9.1 Detection Metrics

The main detector is evaluated as a binary classification problem: grounded versus hallucinated. Weighted F1 is used because it balances performance across classes.


In [ ]:
def evaluate_detection(dataset, weights=(0.2, 0.5, 0.3)):
    """Evaluate hallucination detection on a dataset. Returns metrics dict."""
    nli_preds, pipe_preds, trues = [], [], []

    for sample in dataset:
        trues.append(sample['label'])

        # NLI standalone
        s, label = nli_score(sample['source'], sample['answer'])
        nli_preds.append(1 if label == 'entailment' else 0)

        # Full pipeline
        result = detect_hallucination(sample['source'], sample['answer'], weights)
        pipe_preds.append(1 if result['verdict'] == 'GROUNDED' else 0)

    print("NLI MODEL (DeBERTa-v3-small) — TEST SET")
    print("=" * 50)
    print(classification_report(trues, nli_preds, target_names=['Hallucinated', 'Grounded']))

    print("COMBINED PIPELINE — TEST SET")
    print("=" * 50)
    print(classification_report(trues, pipe_preds, target_names=['Hallucinated', 'Grounded']))

    return {
        'nli': {'preds': nli_preds, 'f1': f1_score(trues, nli_preds, average='weighted')},
        'pipeline': {'preds': pipe_preds, 'f1': f1_score(trues, pipe_preds, average='weighted')},
        'true': trues,
    }

det_results = evaluate_detection(test_data)

### 9.2 Ablation and Weight Search

Ablation tests whether each signal is actually useful. Weight search provides an empirical way to set the fusion weights, but the interpretation should remain cautious because the dataset is small.


In [ ]:
def ablation_eval(dataset, weights, threshold=0.6):
    preds, trues = [], []
    w_ner, w_nli, w_emb = weights
    for s in dataset:
        n = ner_score(s['source'], s['answer'])
        l, _ = nli_score(s['source'], s['answer'])
        e = embedding_score(s['source'], s['answer'])
        total = w_ner + w_nli + w_emb
        if total == 0: total = 1
        c = (w_ner * n + w_nli * l + w_emb * e) / total
        preds.append(1 if c >= threshold else 0)
        trues.append(s['label'])
    return f1_score(trues, preds, average='weighted')

configs = [
    ("NER only",           (1, 0, 0)),
    ("NLI only",           (0, 1, 0)),
    ("Embedding only",     (0, 0, 1)),
    ("NER + NLI",          (0.2, 0.8, 0)),
    ("NLI + Embedding",    (0, 0.6, 0.4)),
    ("Full (0.2/0.5/0.3)", (0.2, 0.5, 0.3)),
]

print("ABLATION STUDY")
print("=" * 50)
print(f"{'Config':<25} {'F1':>8}")
print("-" * 35)
for name, w in configs:
    f1 = ablation_eval(test_data, w)
    print(f"{name:<25} {f1:>8.4f}")

In [ ]:
print("WEIGHT GRID SEARCH")
print("=" * 50)

best_f1, best_w = 0, (0.2, 0.5, 0.3)
opts = [round(x * 0.1, 1) for x in range(11)]
count = 0

for a, b, c in product(opts, repeat=3):
    if abs(a + b + c - 1.0) > 0.01:
        continue
    count += 1
    f1 = ablation_eval(test_data, (a, b, c))
    if f1 > best_f1:
        best_f1 = f1
        best_w = (a, b, c)

print(f"Searched {count} combinations")
print(f"Optimal: NER={best_w[0]}, NLI={best_w[1]}, EMB={best_w[2]}")
print(f"Best F1: {best_f1:.4f}")

In [ ]:
# ============================================================
# E7 — Ablation with delta analysis
# ============================================================
def enhanced_ablation(dataset):
    """Ablation study with explicit performance deltas."""
    configs = [
        ("NER only",           (1, 0, 0)),
        ("NLI only",           (0, 1, 0)),
        ("Embedding only",     (0, 0, 1)),
        ("NER + NLI",          (0.2, 0.8, 0)),
        ("NER + Embedding",    (0.4, 0, 0.6)),
        ("NLI + Embedding",    (0, 0.6, 0.4)),
        ("Full (optimised)",   best_w if 'best_w' in dir() else (0.2, 0.5, 0.3)),
    ]

    results = []
    for name, w in configs:
        f1 = ablation_eval(dataset, w)
        results.append((name, w, f1))

    # Find full model F1
    full_f1 = results[-1][2]

    print("ENHANCED ABLATION STUDY")
    print("=" * 65)
    print(f"{'Configuration':<25} {'Weights':>15} {'F1':>8} {'Δ from Full':>12}")
    print("-" * 65)

    for name, w, f1 in results:
        delta = f1 - full_f1
        delta_str = f"{delta:+.4f}" if name != results[-1][0] else "  (base)"
        w_str = f"({w[0]:.1f},{w[1]:.1f},{w[2]:.1f})"
        print(f"{name:<25} {w_str:>15} {f1:>8.4f} {delta_str:>12}")

    print(f"\nKey findings:")
    nli_only = [r for r in results if r[0] == 'NLI only'][0][2]
    ner_only = [r for r in results if r[0] == 'NER only'][0][2]
    emb_only = [r for r in results if r[0] == 'Embedding only'][0][2]
    print(f"  NLI is the dominant signal (F1={nli_only:.4f})")
    print(f"  NER adds value for entity/number checking (F1={ner_only:.4f})")
    print(f"  Embedding alone is weakest (F1={emb_only:.4f})")

    if full_f1 > nli_only:
        print(f"  Full pipeline improves over NLI-only by {full_f1-nli_only:+.4f}")
    else:
        print(f"  NLI-only slightly outperforms full pipeline ({nli_only-full_f1:+.4f})")
        print(f"  Multi-layer value is in diagnostic richness, not raw F1")

    return results

ablation_enhanced = enhanced_ablation(test_data)

**Ablation interpretation:** Strong NLI performance supports the claim that hallucination detection is primarily a semantic entailment task. NER and embedding similarity add diagnostic and complementary evidence, especially for entities, numbers, and paraphrases.


### 9.3 End-to-End and External Sanity Evaluation

End-to-end evaluation checks whether retrieval, generation, and verification work together. External sanity tests use examples outside the main training construction to reduce the chance that performance is only due to memorised templates.


In [ ]:
def evaluate_pipeline(test_queries, dataset):
    """Full pipeline evaluation: retrieval + generation + detection."""
    print("END-TO-END PIPELINE EVALUATION")
    print("=" * 60)

    # Part A: Retrieval
    ret = evaluate_retrieval(test_queries)

    # Part B: Generation
    gen = evaluate_generation(test_queries[:10])

    # Part C: Detection (on labelled dataset)
    det = evaluate_detection(dataset)

    print("\n--- SUMMARY ---")
    print(f"Retrieval MRR:     {ret['mrr']:.4f}")
    print(f"NLI Detection F1:  {det['nli']['f1']:.4f}")
    print(f"Pipeline F1:       {det['pipeline']['f1']:.4f}")

    return {'retrieval': ret, 'detection': det}

final = evaluate_pipeline(retrieval_queries, test_data)

In [ ]:
# ============================================================
# E3 — End-to-end with semantic correctness
# ============================================================
def semantic_answer_score(answer, source, sbert):
    """Hybrid answer quality score: embedding similarity + keyword overlap."""
    # Semantic similarity
    embs = sbert.encode([answer, source], convert_to_numpy=True)
    sem_sim = float(np.dot(embs[0], embs[1]) /
                    (np.linalg.norm(embs[0]) * np.linalg.norm(embs[1]) + 1e-8))

    # Keyword overlap (Jaccard on content words)
    a_words = set(re.findall(r'\b[a-z]{3,}\b', answer.lower()))
    s_words = set(re.findall(r'\b[a-z]{3,}\b', source.lower()))
    if not a_words or not s_words:
        kw_overlap = 0.0
    else:
        kw_overlap = len(a_words & s_words) / len(a_words | s_words)

    # Hybrid: 70% semantic, 30% keyword
    hybrid = 0.7 * sem_sim + 0.3 * kw_overlap
    return {'semantic': sem_sim, 'keyword': kw_overlap, 'hybrid': hybrid}


def evaluate_e2e(queries, threshold=0.5):
    """End-to-end: retrieval + generation + detection evaluated together."""
    print("END-TO-END EVALUATION (Semantic Scoring)")
    print("=" * 60)

    answer_correct = 0
    detection_correct = 0
    both_correct = 0
    total = 0
    details = []

    for query, expected_cat in queries:
        retrieved = retrieve_passages(query, top_k=5)
        answer, _ = generate_answer(query, retrieved)
        best_source = retrieved[0]['text']

        # Answer quality (semantic)
        score = semantic_answer_score(answer, best_source, sbert_model)
        ans_ok = score['hybrid'] >= threshold

        # Detection quality (is the detection verdict reasonable?)
        det = detect_hallucination(best_source, answer)
        # If answer is semantically close to source, it should be GROUNDED
        # If answer is far from source, it should be HALLUCINATED
        expected_verdict = "GROUNDED" if ans_ok else "HALLUCINATED"
        det_ok = (det['verdict'] == expected_verdict or
                  (det['verdict'] == "PARTIALLY GROUNDED" and ans_ok))

        if ans_ok: answer_correct += 1
        if det_ok: detection_correct += 1
        if ans_ok and det_ok: both_correct += 1
        total += 1

        details.append({
            'query': query, 'answer': answer[:60],
            'hybrid': score['hybrid'], 'ans_ok': ans_ok,
            'verdict': det['verdict'], 'det_ok': det_ok,
        })

    print(f"{'Metric':<30} {'Value':>10}")
    print("-" * 42)
    print(f"{'Answer Correctness':<30} {answer_correct/total:>10.2%} ({answer_correct}/{total})")
    print(f"{'Detection Accuracy':<30} {detection_correct/total:>10.2%} ({detection_correct}/{total})")
    print(f"{'Both Correct (strict)':<30} {both_correct/total:>10.2%} ({both_correct}/{total})")

    print(f"\nSample details:")
    for d in details[:5]:
        ok = "OK" if d['ans_ok'] and d['det_ok'] else "MISS"
        print(f"  [{ok}] Q: {d['query'][:45]}")
        print(f"       A: {d['answer']}...")
        print(f"       hybrid={d['hybrid']:.3f} verdict={d['verdict']}")

    return {'answer_acc': answer_correct/total, 'det_acc': detection_correct/total,
            'both_acc': both_correct/total}

e2e_results = evaluate_e2e(retrieval_queries)

In [ ]:
# ============================================================
# E5 — External test set (NOT derived from training data)
# ============================================================
external_test = [
    # Different phrasing of known topics
    {"source": "Student visa holders can work up to 48 hours per fortnight while their course is in session.",
     "answer": "International students are permitted to be employed for a maximum of 48 hours in any given fortnight during academic terms.",
     "label": 1},
    {"source": "The base application fee for the Student Visa Subclass 500 is AUD 1,600.",
     "answer": "Lodging a student visa application requires payment of one thousand six hundred Australian dollars.",
     "label": 1},
    {"source": "The Partner Visa allows the spouse or de facto partner of an Australian citizen to live in Australia.",
     "answer": "Married couples where one spouse holds Australian citizenship can apply for cohabitation rights through the Partner Visa pathway.",
     "label": 1},
    {"source": "Dual citizenship is permitted in Australia. Applicants are not required to renounce their existing citizenship.",
     "answer": "Australia recognises multiple nationalities and does not mandate surrender of prior citizenship upon naturalisation.",
     "label": 1},
    {"source": "The Working Holiday Visa (Subclass 417) allows young adults aged 18 to 30 from eligible countries to work in Australia for up to 12 months.",
     "answer": "The backpacker visa lets young people between eighteen and thirty years of age take employment in Australia for one year.",
     "label": 1},

    # Subtle hallucinations (harder to detect)
    {"source": "Student visa holders can work up to 48 hours per fortnight while their course is in session.",
     "answer": "Students may work up to 48 hours per week while their course is in session, with additional hours permitted on weekends.",
     "label": 0},  # per week, not per fortnight
    {"source": "The base application fee for the Student Visa Subclass 500 is AUD 1,600.",
     "answer": "The student visa application fee is AUD 1,600, which is fully refundable if the application is unsuccessful.",
     "label": 0},  # not refundable
    {"source": "All student visa holders must maintain Overseas Student Health Cover (OSHC) for the entire duration of their stay.",
     "answer": "Students must have OSHC for the first 12 months of their stay, after which they can switch to Medicare coverage.",
     "label": 0},  # not just 12 months
    {"source": "To be eligible for Australian citizenship by conferral, applicants must have lived in Australia for at least four years.",
     "answer": "Citizenship applicants need three years of continuous residence in Australia to be eligible for conferral.",
     "label": 0},  # 4 years not 3
    {"source": "The Partner Visa allows the spouse or de facto partner of an Australian citizen to live in Australia.",
     "answer": "The Partner Visa is available to engaged couples who plan to marry within 24 months of arriving in Australia.",
     "label": 0},  # that's the Prospective Marriage Visa, not Partner Visa
    {"source": "The Visitor Visa Subclass 600 can be granted for stays of 3, 6, or 12 months.",
     "answer": "The visitor visa allows stays of 3, 6, 12, or 24 months depending on the applicant's profile.",
     "label": 0},  # no 24 month option
    {"source": "Bridging Visa A (BVA) is automatically granted to visa applicants who hold a valid substantive visa when they apply for a new visa.",
     "answer": "A Bridging Visa A is granted upon request to anyone present in Australia, regardless of their current visa status.",
     "label": 0},  # must hold valid substantive visa
    {"source": "The Skilled Independent Visa (Subclass 189) requires a minimum points score of 65.",
     "answer": "The Subclass 189 visa requires applicants to score at least 60 points on the skilled migration points test.",
     "label": 0},  # 65 not 60
    {"source": "Working Holiday Visa holders can work for the same employer for up to 6 months.",
     "answer": "Working Holiday Visa holders can work for the same employer for up to 12 months without restrictions.",
     "label": 0},  # 6 months not 12
    {"source": "Most visa applications must be submitted online through ImmiAccount.",
     "answer": "All visa applications must be submitted in person at an Australian embassy or consulate. Online applications are not accepted.",
     "label": 0},  # online, not in person
]

print("EXTERNAL SANITY TEST (15 unseen queries)")
print("=" * 60)

ext_nli_preds, ext_pipe_preds, ext_true = [], [], []

for i, sample in enumerate(external_test):
    s_nli, label = nli_score(sample['source'], sample['answer'])
    nli_pred = 1 if label == 'entailment' else 0

    det = detect_hallucination(sample['source'], sample['answer'])
    pipe_pred = 1 if det['verdict'] == 'GROUNDED' else 0

    ext_true.append(sample['label'])
    ext_nli_preds.append(nli_pred)
    ext_pipe_preds.append(pipe_pred)

    true_label = "GROUNDED" if sample['label'] == 1 else "HALLUCINATED"
    nli_ok = "✓" if nli_pred == sample['label'] else "✗"
    pipe_ok = "✓" if pipe_pred == sample['label'] else "✗"
    print(f"  [{true_label:>12s}] NLI:{nli_ok} Pipe:{pipe_ok} | {sample['answer'][:65]}...")

ext_nli_f1 = f1_score(ext_true, ext_nli_preds, average='weighted')
ext_pipe_f1 = f1_score(ext_true, ext_pipe_preds, average='weighted')

print(f"\nExternal Test Results:")
print(f"  NLI F1:      {ext_nli_f1:.4f}")
print(f"  Pipeline F1: {ext_pipe_f1:.4f}")
print(f"  NLI Accuracy:      {accuracy_score(ext_true, ext_nli_preds):.4f}")
print(f"  Pipeline Accuracy: {accuracy_score(ext_true, ext_pipe_preds):.4f}")

print(f"\nNote: External test uses SUBTLE hallucinations (single-fact changes)")
print(f"which are harder than the main evaluation set's obvious contradictions.")

**External-test interpretation:** These examples are not a large benchmark. They are a sanity check for paraphrase sensitivity and subtle factual errors.


### 9.4 Model Compression and Distillation

The distillation comparison examines whether a smaller NLI model can preserve useful detection performance while reducing latency. This is relevant if the verifier is later deployed in resource-constrained settings.


In [ ]:
distil_tok = AutoTokenizer.from_pretrained("cross-encoder/nli-distilroberta-base")
distil_mod = AutoModelForSequenceClassification.from_pretrained(
    "cross-encoder/nli-distilroberta-base").to(device)

def bench_nli(model, tokenizer, dataset, name):
    preds, trues = [], []
    t0 = time.time()
    for s in dataset:
        inp = tokenizer(s['source'], s['answer'], return_tensors="pt",
                        truncation=True, max_length=512, padding=True).to(device)
        with torch.no_grad():
            p = 1 if torch.argmax(F.softmax(model(**inp).logits, dim=1)[0]).item() == 1 else 0
        preds.append(p)
        trues.append(s['label'])
    elapsed = time.time() - t0
    params = sum(p.numel() for p in model.parameters())
    f1 = f1_score(trues, preds, average='weighted')
    ms = elapsed / len(dataset) * 1000
    print(f"{name}: F1={f1:.4f}, {ms:.1f}ms/sample, {params:,} params")
    return {'f1': f1, 'ms': ms, 'params': params}

print("DISTILLATION COMPARISON")
print("=" * 50)
full_r = bench_nli(nli_model, nli_tokenizer, test_data, "DeBERTa (Full)")
dist_r = bench_nli(distil_mod, distil_tok, test_data, "DistilRoBERTa")
print(f"Speedup: {full_r['ms']/max(dist_r['ms'], 0.1):.2f}x")

### 9.5 Repeated Runs, Calibration, and Statistical Testing

Repeated runs report sensitivity to random train/test splits. Calibration checks whether confidence scores align with empirical correctness. Bootstrap testing estimates whether observed differences are stable enough to discuss, while avoiding overly strong claims from a small dataset.


In [ ]:
# ============================================================
# E6 — Multiple runs with different seeds
# ============================================================
def run_multiple_experiments(dataset, n_runs=3):
    """Run evaluation multiple times and report mean ± std."""
    print(f"STATISTICAL EVALUATION ({n_runs} runs)")
    print("=" * 60)

    nli_f1s, pipe_f1s = [], []

    for run in range(n_runs):
        # Different split each run
        seed = SEED + run * 17
        _, run_test = prepare_dataset(dataset, test_size=0.3,
                                      augment_multiplier=4, seed=seed)

        nli_p, pipe_p, trues = [], [], []
        for s in run_test:
            trues.append(s['label'])
            _, label = nli_score(s['source'], s['answer'])
            nli_p.append(1 if label == 'entailment' else 0)

            det = detect_hallucination(s['source'], s['answer'])
            pipe_p.append(1 if det['verdict'] == 'GROUNDED' else 0)

        nli_f1s.append(f1_score(trues, nli_p, average='weighted'))
        pipe_f1s.append(f1_score(trues, pipe_p, average='weighted'))
        print(f"  Run {run+1}: NLI F1={nli_f1s[-1]:.4f}, Pipeline F1={pipe_f1s[-1]:.4f}")

    print(f"\nAggregated Results:")
    print(f"  NLI Detection F1:      {np.mean(nli_f1s):.4f} ± {np.std(nli_f1s):.4f}")
    print(f"  Pipeline Detection F1: {np.mean(pipe_f1s):.4f} ± {np.std(pipe_f1s):.4f}")

    return {
        'nli_mean': np.mean(nli_f1s), 'nli_std': np.std(nli_f1s),
        'pipe_mean': np.mean(pipe_f1s), 'pipe_std': np.std(pipe_f1s),
    }

stat_results = run_multiple_experiments(hallucination_dataset, n_runs=3)

In [ ]:
# ════════════════════════════════════════════════════════════════
# U6 — CONFIDENCE CALIBRATION + ECE
# ════════════════════════════════════════════════════════════════

def confidence_calibration(dataset, weights=(0.2, 0.5, 0.3), n_bins=10):
    """Compute Expected Calibration Error (ECE) and reliability data."""
    confidences = []
    accuracies = []

    for sample in dataset:
        result = detect_hallucination(sample['source'], sample['answer'], weights)
        conf = result['combined']
        pred = 1 if result['verdict'] == 'GROUNDED' else 0
        correct = int(pred == sample['label'])
        confidences.append(conf)
        accuracies.append(correct)

    confidences = np.array(confidences)
    accuracies = np.array(accuracies)

    # ECE computation
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_data = []

    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        mask = (confidences >= lo) & (confidences < hi)
        if mask.sum() == 0:
            bin_data.append({'lo': lo, 'hi': hi, 'avg_conf': 0, 'avg_acc': 0, 'count': 0})
            continue
        avg_conf = confidences[mask].mean()
        avg_acc = accuracies[mask].mean()
        weight = mask.sum() / len(confidences)
        ece += weight * abs(avg_acc - avg_conf)
        bin_data.append({'lo': lo, 'hi': hi, 'avg_conf': avg_conf, 'avg_acc': avg_acc, 'count': int(mask.sum())})

    print("CONFIDENCE CALIBRATION")
    print("=" * 60)
    print(f"Expected Calibration Error (ECE): {ece:.4f}")
    print(f"  (Lower is better; < 0.05 is well-calibrated)")
    print(f"\n{'Bin':<12} {'Avg Conf':>10} {'Avg Acc':>10} {'Count':>8} {'|Gap|':>8}")
    print("-" * 55)
    for b in bin_data:
        if b['count'] > 0:
            gap = abs(b['avg_acc'] - b['avg_conf'])
            print(f"[{b['lo']:.1f}-{b['hi']:.1f}] {b['avg_conf']:>10.3f} {b['avg_acc']:>10.3f} {b['count']:>8} {gap:>8.3f}")

    return {'ece': ece, 'bins': bin_data, 'confidences': confidences, 'accuracies': accuracies}

calibration = confidence_calibration(test_data_v5)


In [ ]:
# ════════════════════════════════════════════════════════════════
# U9 — STATISTICAL SIGNIFICANCE TESTING (Bootstrap)
# ════════════════════════════════════════════════════════════════

def bootstrap_significance(scores_a, scores_b, n_bootstrap=1000, confidence=0.95):
    """Bootstrap test for statistical significance between two sets of scores.
    scores_a, scores_b: arrays of per-sample correctness (0 or 1).
    Returns: p-value, confidence interval for difference.
    """
    np.random.seed(42)
    n = len(scores_a)
    observed_diff = np.mean(scores_a) - np.mean(scores_b)

    # Bootstrap under null (combined)
    combined = np.concatenate([scores_a, scores_b])
    diffs = []
    for _ in range(n_bootstrap):
        perm = np.random.permutation(combined)
        diff = np.mean(perm[:n]) - np.mean(perm[n:])
        diffs.append(diff)

    diffs = np.array(diffs)
    p_value = np.mean(np.abs(diffs) >= np.abs(observed_diff))

    # Bootstrap CI for difference
    boot_diffs = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, size=n, replace=True)
        boot_diffs.append(np.mean(scores_a[idx]) - np.mean(scores_b[idx]))
    boot_diffs = np.array(boot_diffs)
    alpha = 1 - confidence
    ci_lo = np.percentile(boot_diffs, 100 * alpha / 2)
    ci_hi = np.percentile(boot_diffs, 100 * (1 - alpha / 2))

    return {'observed_diff': observed_diff, 'p_value': p_value,
            'ci_lo': ci_lo, 'ci_hi': ci_hi}


def run_significance_tests(test_data):
    """Run bootstrap significance tests for key comparisons."""
    print("STATISTICAL SIGNIFICANCE TESTS (Bootstrap, n=1000)")
    print("=" * 65)

    trues = np.array([d['label'] for d in test_data])

    # NLI vs Full Pipeline
    nli_correct = np.array([
        int((1 if nli_score(d['source'], d['answer'])[1] == 'entailment' else 0) == d['label'])
        for d in test_data
    ])
    pipe_correct = np.array([
        int((1 if detect_hallucination(d['source'], d['answer'],
             threshold=OPTIMAL_THRESHOLD)['verdict'] == 'GROUNDED' else 0) == d['label'])
        for d in test_data
    ])

    # Embedding-only vs Pipeline
    emb_correct = np.array([
        int((1 if embedding_score(d['source'], d['answer']) >= 0.7 else 0) == d['label'])
        for d in test_data
    ])

    comparisons = [
        ("Full Pipeline vs NLI-only", pipe_correct, nli_correct),
        ("Full Pipeline vs Emb-only", pipe_correct, emb_correct),
    ]

    print(f"{'Comparison':<35} {'Diff':>8} {'p-value':>10} {'95% CI':>20} {'Sig?':>6}")
    print("-" * 85)
    for name, a, b in comparisons:
        result = bootstrap_significance(a, b)
        sig = "Yes" if result['p_value'] < 0.05 else "No"
        ci_str = f"[{result['ci_lo']:+.3f}, {result['ci_hi']:+.3f}]"
        print(f"{name:<35} {result['observed_diff']:>+8.3f} {result['p_value']:>10.4f} {ci_str:>20} {sig:>6}")

    # Retrieval: SBERT vs Hybrid
    if 'SBERT Only' in retrieval_comparison and 'Hybrid + Rerank' in retrieval_comparison:
        rrs_sbert = np.array(retrieval_comparison['SBERT Only']['rrs'])
        rrs_hybrid = np.array(retrieval_comparison['Hybrid + Rerank']['rrs'])
        ret_result = bootstrap_significance(rrs_hybrid, rrs_sbert)
        sig = "Yes" if ret_result['p_value'] < 0.05 else "No"
        ci_str = f"[{ret_result['ci_lo']:+.3f}, {ret_result['ci_hi']:+.3f}]"
        print(f"{'Hybrid+Rerank vs SBERT (MRR)':<35} {ret_result['observed_diff']:>+8.3f} {ret_result['p_value']:>10.4f} {ci_str:>20} {sig:>6}")

    return comparisons

sig_tests = run_significance_tests(test_data_v5)


## 10. Experimental Results

This section gathers the most important outputs in one place for easy marker review. The key story is whether semantic verification improves hallucination detection and whether the system remains interpretable on hard negatives.


### 10.1 Core Detection Diagnostics

This figure places the key baseline and verifier results beside the confusion matrices. It is the fastest way to inspect whether errors are concentrated in hallucinated or grounded examples.


In [ ]:
# Key diagnostic plots for the main detection experiment.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("HalluciGuard: Core Detection Diagnostics", fontsize=15, fontweight="bold")

# 1. Baseline training loss
axes[0, 0].plot(cnn_losses, label="Text CNN", color="#e74c3c", lw=2)
axes[0, 0].plot(lstm_losses, label="BiLSTM", color="#3498db", lw=2)
axes[0, 0].set_title("Baseline Training Loss", fontweight="bold")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. F1 comparison across shallow baselines and semantic verification models
models_f1 = {
    "Text CNN": f1_score(cnn_true, cnn_pred, average="weighted"),
    "BiLSTM": f1_score(lstm_true, lstm_pred, average="weighted"),
    "NLI": det_results["nli"]["f1"],
    "Pipeline": det_results["pipeline"]["f1"],
}
bars = axes[0, 1].bar(models_f1.keys(), models_f1.values(),
                      color=["#e74c3c", "#3498db", "#2ecc71", "#34495e"])
for bar, value in zip(bars, models_f1.values()):
    axes[0, 1].text(bar.get_x() + bar.get_width() / 2, value + 0.02,
                    f"{value:.3f}", ha="center", fontsize=9, fontweight="bold")
axes[0, 1].set_title("Weighted F1 Comparison", fontweight="bold")
axes[0, 1].set_ylim(0, 1.15)
axes[0, 1].grid(alpha=0.3, axis="y")

# 3. NLI confusion matrix
cm_nli = confusion_matrix(det_results["true"], det_results["nli"]["preds"])
sns.heatmap(cm_nli, annot=True, fmt="d", cmap="Blues", ax=axes[1, 0],
            xticklabels=["Hallucinated", "Grounded"],
            yticklabels=["Hallucinated", "Grounded"])
axes[1, 0].set_title("NLI Confusion Matrix", fontweight="bold")
axes[1, 0].set_xlabel("Predicted")
axes[1, 0].set_ylabel("True")

# 4. Full pipeline confusion matrix
cm_pipe = confusion_matrix(det_results["true"], det_results["pipeline"]["preds"])
sns.heatmap(cm_pipe, annot=True, fmt="d", cmap="Greens", ax=axes[1, 1],
            xticklabels=["Hallucinated", "Grounded"],
            yticklabels=["Hallucinated", "Grounded"])
axes[1, 1].set_title("Full Pipeline Confusion Matrix", fontweight="bold")
axes[1, 1].set_xlabel("Predicted")
axes[1, 1].set_ylabel("True")

plt.tight_layout()
plt.show()


**Confusion-matrix interpretation:** False grounded predictions are the riskiest error because they allow unsupported information through. False hallucinated predictions are also undesirable, but they are usually safer in an advisory system.


### 10.2 v5 Evaluation Dashboard

The v5 dashboard groups threshold tuning, calibration, baseline comparison, category-wise accuracy, retrieval comparison, and ablation deltas in one place.


In [ ]:
# ════════════════════════════════════════════════════════════════
# U8 — IMPROVED VISUALISATIONS
# ════════════════════════════════════════════════════════════════

def plot_v5_dashboard(threshold_curve, calibration, baseline_results,
                      cat_results, retrieval_comparison):
    """Generate comprehensive v5 visualisation dashboard."""
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('HalluciGuard v5 — Evaluation Dashboard', fontsize=16, fontweight='bold', y=1.02)

    # 1. Threshold optimisation curve
    ax = axes[0, 0]
    ts = [r['threshold'] for r in threshold_curve]
    f1s = [r['f1'] for r in threshold_curve]
    precs = [r['precision'] for r in threshold_curve]
    recs = [r['recall'] for r in threshold_curve]
    ax.plot(ts, f1s, 'b-', lw=2, label='F1')
    ax.plot(ts, precs, 'g--', lw=1.5, label='Precision')
    ax.plot(ts, recs, 'r--', lw=1.5, label='Recall')
    best_idx = np.argmax(f1s)
    ax.axvline(x=ts[best_idx], color='k', ls=':', alpha=0.5)
    ax.scatter([ts[best_idx]], [f1s[best_idx]], c='blue', s=100, zorder=5)
    ax.annotate(f'Best: t={ts[best_idx]:.2f}\nF1={f1s[best_idx]:.3f}',
                xy=(ts[best_idx], f1s[best_idx]), xytext=(ts[best_idx]+0.1, f1s[best_idx]-0.1),
                fontsize=8, arrowprops=dict(arrowstyle='->', color='gray'))
    ax.set_title('Threshold Optimisation', fontweight='bold')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # 2. Confidence calibration (reliability diagram)
    ax = axes[0, 1]
    bins = calibration['bins']
    bin_confs = [b['avg_conf'] for b in bins if b['count'] > 0]
    bin_accs = [b['avg_acc'] for b in bins if b['count'] > 0]
    bin_counts = [b['count'] for b in bins if b['count'] > 0]
    ax.bar(range(len(bin_confs)), bin_accs, alpha=0.6, color='#3498db', label='Accuracy')
    ax.plot(range(len(bin_confs)), bin_confs, 'r-o', lw=2, label='Confidence')
    ax.plot([0, len(bin_confs)-1], [0, 1], 'k--', alpha=0.3, label='Perfect calibration')
    ax.set_title(f'Reliability Diagram (ECE={calibration["ece"]:.3f})', fontweight='bold')
    ax.set_xlabel('Confidence Bin')
    ax.set_ylabel('Fraction')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # 3. Baseline comparison
    ax = axes[0, 2]
    names = list(baseline_results.keys())
    vals = list(baseline_results.values())
    colors = ['#2ecc71' if n == 'Full Pipeline' else '#e74c3c' if v < 0.5
              else '#3498db' for n, v in zip(names, vals)]
    sorted_pairs = sorted(zip(names, vals, colors), key=lambda x: x[1])
    names_s, vals_s, colors_s = zip(*sorted_pairs)
    bars = ax.barh(range(len(names_s)), vals_s, color=colors_s)
    ax.set_yticks(range(len(names_s)))
    ax.set_yticklabels(names_s, fontsize=8)
    for bar, v in zip(bars, vals_s):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{v:.3f}', va='center', fontsize=8)
    ax.set_title('Baseline Comparison (F1)', fontweight='bold')
    ax.set_xlim(0, 1.15)
    ax.grid(alpha=0.3, axis='x')

    # 4. Category-wise accuracy
    ax = axes[1, 0]
    cat_names = sorted(cat_results.keys())
    cat_accs = [cat_results[c]['correct'] / cat_results[c]['total']
                if cat_results[c]['total'] > 0 else 0 for c in cat_names]
    cat_cols = ['#e74c3c' if a < 0.7 else '#f39c12' if a < 0.85 else '#2ecc71' for a in cat_accs]
    bars = ax.bar(range(len(cat_names)), cat_accs, color=cat_cols)
    ax.set_xticks(range(len(cat_names)))
    ax.set_xticklabels(cat_names, rotation=30, ha='right', fontsize=8)
    for bar, v in zip(bars, cat_accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{v:.2f}', ha='center', fontsize=8)
    ax.set_title('Category-wise Accuracy', fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.3, axis='y')

    # 5. Retrieval comparison
    ax = axes[1, 1]
    ret_methods = list(retrieval_comparison.keys())
    ret_mrrs = [retrieval_comparison[m]['mrr'] for m in ret_methods]
    ret_r1 = [retrieval_comparison[m]['recall'][1] for m in ret_methods]
    x = np.arange(len(ret_methods))
    w = 0.35
    ax.bar(x - w/2, ret_mrrs, w, label='MRR', color='#3498db')
    ax.bar(x + w/2, ret_r1, w, label='Recall@1', color='#e67e22')
    ax.set_xticks(x)
    ax.set_xticklabels(ret_methods, rotation=20, ha='right', fontsize=7)
    ax.set_title('Retrieval Method Comparison', fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, axis='y')

    # 6. Ablation deltas (from v4 enhanced ablation if available)
    ax = axes[1, 2]
    ablation_names = ['NER only', 'NLI only', 'Emb only', 'NER+NLI', 'NLI+Emb', 'Full']
    ablation_f1s = [
        ablation_eval(test_data_v5, (1,0,0)),
        ablation_eval(test_data_v5, (0,1,0)),
        ablation_eval(test_data_v5, (0,0,1)),
        ablation_eval(test_data_v5, (0.2,0.8,0)),
        ablation_eval(test_data_v5, (0,0.6,0.4)),
        ablation_eval(test_data_v5, (0.2,0.5,0.3)),
    ]
    full_f1 = ablation_f1s[-1]
    deltas = [f - full_f1 for f in ablation_f1s]
    colors = ['#2ecc71' if d >= 0 else '#e74c3c' for d in deltas]
    ax.bar(range(len(ablation_names)), deltas, color=colors)
    ax.set_xticks(range(len(ablation_names)))
    ax.set_xticklabels(ablation_names, rotation=30, ha='right', fontsize=8)
    ax.axhline(y=0, color='k', lw=0.8)
    ax.set_title('Ablation Deltas (vs Full Pipeline)', fontweight='bold')
    ax.set_ylabel('ΔF1')
    ax.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('halluciguard_v5_dashboard.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: halluciguard_v5_dashboard.png")

plot_v5_dashboard(threshold_curve, calibration, baseline_results,
                  cat_results, retrieval_comparison)


**Dashboard interpretation:** Read the dashboard from left to right: decision threshold, confidence reliability, baseline comparison, category robustness, retrieval quality, and contribution of each verification signal.


### 10.3 Consolidated Result Summary

The printed dashboard summarises the main v4 evaluation metrics in a report-friendly format. These values are useful for the project report and 4-minute presentation.


In [ ]:
# ============================================================
# E8 — Final consolidated output
# ============================================================
print()
print("╔" + "═"*58 + "╗")
print("║" + "  HALLUCIGUARD — FINAL EVALUATION DASHBOARD  ".center(58) + "║")
print("╠" + "═"*58 + "╣")

print("║" + "  RETRIEVAL".ljust(58) + "║")
print("║" + f"    MRR (SBERT):          {ret_comparison['mrr_sbert']:.4f}".ljust(58) + "║")
print("║" + f"    MRR (+ Re-ranking):   {ret_comparison['mrr_reranked']:.4f}".ljust(58) + "║")
print("║" + f"    Recall@5:             {ret_metrics['recall'].get(5, 'N/A')}".ljust(58) + "║")

print("╠" + "═"*58 + "╣")
print("║" + "  GENERATION".ljust(58) + "║")
gen_fails = sum(1 for r in gen_results if r['fail'])
gen_total = len(gen_results)
gen_avg = np.mean([r['words'] for r in gen_results])
print("║" + f"    Failure rate:         {gen_fails}/{gen_total} ({gen_fails/gen_total:.0%})".ljust(58) + "║")
print("║" + f"    Avg answer length:    {gen_avg:.1f} words".ljust(58) + "║")

print("╠" + "═"*58 + "╣")
print("║" + "  HALLUCINATION DETECTION".ljust(58) + "║")
print("║" + f"    NLI F1 (test):        {det_results['nli']['f1']:.4f}".ljust(58) + "║")
print("║" + f"    Pipeline F1 (test):   {det_results['pipeline']['f1']:.4f}".ljust(58) + "║")
print("║" + f"    NLI F1 (mean±std):    {stat_results['nli_mean']:.4f} ± {stat_results['nli_std']:.4f}".ljust(58) + "║")

print("╠" + "═"*58 + "╣")
print("║" + "  END-TO-END PIPELINE".ljust(58) + "║")
print("║" + f"    Answer correctness:   {e2e_results['answer_acc']:.2%}".ljust(58) + "║")
print("║" + f"    Detection accuracy:   {e2e_results['det_acc']:.2%}".ljust(58) + "║")
print("║" + f"    Both correct:         {e2e_results['both_acc']:.2%}".ljust(58) + "║")

print("╠" + "═"*58 + "╣")
print("║" + "  EXTERNAL VALIDATION".ljust(58) + "║")
print("║" + f"    NLI F1 (external):    {ext_nli_f1:.4f}".ljust(58) + "║")
print("║" + f"    Pipeline F1 (ext):    {ext_pipe_f1:.4f}".ljust(58) + "║")

print("╠" + "═"*58 + "╣")
print("║" + "  BASELINES (overfitting diagnostic)".ljust(58) + "║")
print("║" + f"    CNN:  train={cnn_overfit['train']:.2f} test={cnn_overfit['test']:.2f} [{cnn_overfit['flag']}]".ljust(58) + "║")
print("║" + f"    LSTM: train={lstm_overfit['train']:.2f} test={lstm_overfit['test']:.2f} [{lstm_overfit['flag']}]".ljust(58) + "║")

print("╠" + "═"*58 + "╣")
print("║" + "  DATA INTEGRITY".ljust(58) + "║")
print("║" + f"    Cross-split leakage:  {split_validation['leakage_risk']}".ljust(58) + "║")
print("║" + f"    Near-duplicates:      {split_validation['near_dupes']}".ljust(58) + "║")

print("╚" + "═"*58 + "╝")
print()
print("GitHub: https://github.com/Devarsh-04/halluci-guard-nlp")

## 11. Robustness & Error Analysis

Robustness analysis asks whether the system behaves sensibly on adversarial phrasing, out-of-domain questions, paraphrases, and difficult hallucination categories. Error analysis then identifies which failure modes remain.


### 11.1 Robustness Probes

These probes are qualitative and diagnostic rather than a replacement for a large benchmark. They help reveal whether the pipeline is brittle under unusual phrasing or unsupported questions.


In [ ]:
robustness_tests = [
    {"type":"adversarial","query":"Is it true that student visa holders can work unlimited hours?"},
    {"type":"adversarial","query":"I heard the student visa only costs $500. Is that correct?"},
    {"type":"adversarial","query":"Can I get permanent residency automatically after studying?"},
    {"type":"adversarial","query":"My friend said OSHC is optional. Is that right?"},
    {"type":"adversarial","query":"Do I need to give up my citizenship if I become Australian?"},
    {"type":"adversarial","query":"Is it true visitors can work 20 hours a week?"},
    {"type":"adversarial","query":"Can anyone give me immigration advice for a fee?"},
    {"type":"out_of_domain","query":"What is the weather like in Melbourne?"},
    {"type":"out_of_domain","query":"How do I open a bank account in Australia?"},
    {"type":"out_of_domain","query":"What is the best university for CS in Australia?"},
    {"type":"out_of_domain","query":"How much does rent cost in Sydney?"},
    {"type":"out_of_domain","query":"What is the minimum wage in Australia?"},
    {"type":"out_of_domain","query":"How do I get an Australian driving licence?"},
    {"type":"out_of_domain","query":"What vaccinations do I need for Australia?"},
    {"type":"paraphrased","query":"How much time can an international student work every two weeks?"},
    {"type":"paraphrased","query":"What health insurance do I need as a student?"},
    {"type":"paraphrased","query":"If my visa gets rejected, can I challenge the decision?"},
    {"type":"paraphrased","query":"What's the process to become an Aussie citizen?"},
    {"type":"paraphrased","query":"How long can tourists stay in Australia?"},
    {"type":"paraphrased","query":"Do Kiwis need a visa to live in Australia?"},
]

print(f"ROBUSTNESS TESTING ({len(robustness_tests)} queries)")
print("=" * 70)

rob_results = []
for t in robustness_tests:
    retrieved = retrieve_passages(t['query'], top_k=5)
    answer, _ = generate_answer(t['query'], retrieved)
    result = detect_hallucination(retrieved[0]['text'], answer)
    print(f"[{t['type']:>12s}] {result['verdict']:>20s} | Q: {t['query'][:50]}")
    print(f"               A: {answer[:60]}...")
    rob_results.append({**t, 'answer': answer, 'verdict': result['verdict'],
                        'score': result['combined']})

# Summary
print("\nROBUSTNESS SUMMARY")
print("=" * 50)
for rtype in ['adversarial', 'out_of_domain', 'paraphrased']:
    sub = [r for r in rob_results if r['type'] == rtype]
    verdicts = Counter(r['verdict'] for r in sub)
    print(f"  {rtype}: {dict(verdicts)}")

### 11.2 Systematic Failure Analysis

This analysis inspects false positives and false negatives from the main test set. It helps connect model errors to concrete examples rather than only aggregate scores.


In [ ]:
print("FAILURE ANALYSIS")
print("=" * 60)

fp, fn = [], []
for i, s in enumerate(test_data):
    r = detect_hallucination(s['source'], s['answer'])
    pred = 1 if r['verdict'] == 'GROUNDED' else 0
    if s['label'] == 1 and pred == 0:
        fp.append((i, s, r))
    elif s['label'] == 0 and pred == 1:
        fn.append((i, s, r))

print(f"\nFalse Positives (grounded flagged as hallucinated): {len(fp)}")
for idx, s, r in fp[:3]:
    print(f"  NER={r['ner']:.2f} NLI={r['nli']:.2f}({r['nli_label']}) EMB={r['emb']:.2f}")
    print(f"  Answer: {s['answer'][:80]}...")

print(f"\nFalse Negatives (hallucinated passed): {len(fn)}")
for idx, s, r in fn[:3]:
    print(f"  NER={r['ner']:.2f} NLI={r['nli']:.2f}({r['nli_label']}) EMB={r['emb']:.2f}")
    print(f"  Answer: {s['answer'][:80]}...")

### 11.3 Error Taxonomy

The taxonomy groups errors into interpretable categories such as numerical hallucination, entity confusion, retrieval miss, paraphrase confusion, and unsupported inference.


In [ ]:
# ════════════════════════════════════════════════════════════════
# U5 — ERROR TAXONOMY ANALYSIS
# ════════════════════════════════════════════════════════════════

def error_taxonomy(dataset, weights=(0.2, 0.5, 0.3), threshold=None):
    """Categorise detection errors into an error taxonomy."""
    if threshold is None:
        threshold = OPTIMAL_THRESHOLD

    taxonomy = {
        'entity_confusion': [],
        'numerical_hallucination': [],
        'negation_failure': [],
        'retrieval_miss': [],
        'paraphrase_confusion': [],
        'unsupported_inference': [],
    }

    for sample in dataset:
        result = detect_hallucination(sample['source'], sample['answer'], weights, threshold)
        pred = 1 if result['verdict'] == 'GROUNDED' else 0

        if pred == sample['label']:
            continue  # Skip correct predictions

        # Classify the error
        error_info = {
            'source': sample['source'][:80],
            'answer': sample['answer'][:80],
            'true_label': 'GROUNDED' if sample['label'] == 1 else 'HALLUCINATED',
            'pred_label': result['verdict'],
            'scores': result,
            'difficulty': sample.get('difficulty', 'unknown'),
        }

        # Heuristic classification of error type
        src_nums = set(re.findall(r'\b\d+(?:,\d+)*(?:\.\d+)?\b', sample['source']))
        ans_nums = set(re.findall(r'\b\d+(?:,\d+)*(?:\.\d+)?\b', sample['answer']))

        if src_nums != ans_nums and (ans_nums - src_nums):
            taxonomy['numerical_hallucination'].append(error_info)
        elif any(neg in sample['answer'].lower() for neg in ['not ', 'no ', 'cannot', "don't", "doesn't", 'never']):
            if sample['label'] == 1:  # Grounded but predicted hallucinated
                taxonomy['negation_failure'].append(error_info)
            else:
                taxonomy['unsupported_inference'].append(error_info)
        elif result['emb'] > 0.7 and result['nli'] < 0.3:
            taxonomy['paraphrase_confusion'].append(error_info)
        elif result['ner'] < 0.5:
            taxonomy['entity_confusion'].append(error_info)
        else:
            taxonomy['unsupported_inference'].append(error_info)

    total_errors = sum(len(v) for v in taxonomy.values())

    print("ERROR TAXONOMY ANALYSIS")
    print("=" * 60)
    print(f"Total errors: {total_errors}")
    print(f"\n{'Error Type':<30} {'Count':>8} {'Percent':>10}")
    print("-" * 50)
    for etype, errors in sorted(taxonomy.items(), key=lambda x: -len(x[1])):
        pct = len(errors) / total_errors * 100 if total_errors > 0 else 0
        print(f"{etype:<30} {len(errors):>8} {pct:>9.1f}%")
        if errors:
            ex = errors[0]
            print(f"  Example: True={ex['true_label']}, Pred={ex['pred_label']}")
            print(f"    Source: {ex['source']}...")
            print(f"    Answer: {ex['answer']}...")

    return taxonomy

error_tax = error_taxonomy(test_data_v5)


**Error-analysis interpretation:** The most important discussion is not simply how many errors occur, but what kinds of errors remain and whether they are acceptable for the target use case.


### 11.4 Final v5 Consolidated Dashboard

This dashboard is placed after the taxonomy computation because it summarises both headline metrics and remaining error categories.


In [ ]:
# ════════════════════════════════════════════════════════════════
# U11 — FINAL CONSOLIDATED DASHBOARD (v5)
# ════════════════════════════════════════════════════════════════

def print_v5_dashboard():
    """Print the final v5 evaluation dashboard."""
    # Re-evaluate on v5 test set with optimal threshold
    nli_preds_v5, pipe_preds_v5, trues_v5 = [], [], []
    for s in test_data_v5:
        trues_v5.append(s['label'])
        _, label = nli_score(s['source'], s['answer'])
        nli_preds_v5.append(1 if label == 'entailment' else 0)
        det = detect_hallucination(s['source'], s['answer'], threshold=OPTIMAL_THRESHOLD)
        pipe_preds_v5.append(1 if det['verdict'] == 'GROUNDED' else 0)

    nli_f1_v5 = f1_score(trues_v5, nli_preds_v5, average='weighted')
    pipe_f1_v5 = f1_score(trues_v5, pipe_preds_v5, average='weighted')

    # Multi-run on v5 dataset
    nli_f1s, pipe_f1s = [], []
    for run in range(3):
        seed = SEED + run * 17
        _, run_test = prepare_dataset(hallucination_dataset_v5, test_size=0.3, seed=seed)
        nli_p, pipe_p, trues_r = [], [], []
        for s in run_test:
            trues_r.append(s['label'])
            _, label = nli_score(s['source'], s['answer'])
            nli_p.append(1 if label == 'entailment' else 0)
            det = detect_hallucination(s['source'], s['answer'], threshold=OPTIMAL_THRESHOLD)
            pipe_p.append(1 if det['verdict'] == 'GROUNDED' else 0)
        nli_f1s.append(f1_score(trues_r, nli_p, average='weighted'))
        pipe_f1s.append(f1_score(trues_r, pipe_p, average='weighted'))

    best_ret = max(retrieval_comparison.items(), key=lambda x: x[1]['mrr'])

    print()
    print("╔" + "═"*62 + "╗")
    print("║" + "  HALLUCIGUARD v5 — FINAL EVALUATION DASHBOARD  ".center(62) + "║")
    print("╠" + "═"*62 + "╣")
    print("║" + f"  Dataset: {len(hallucination_dataset_v5)} samples ({grounded_v5}G / {hallucinated_v5}H)".ljust(62) + "║")
    print("║" + f"  Difficulty categories: {len(diff_counts)}".ljust(62) + "║")
    print("╠" + "═"*62 + "╣")
    print("║" + "  RETRIEVAL".ljust(62) + "║")
    print("║" + f"    Best method: {best_ret[0]}".ljust(62) + "║")
    print("║" + f"    MRR:         {best_ret[1]['mrr']:.4f}".ljust(62) + "║")
    print("║" + f"    Recall@1:    {best_ret[1]['recall'][1]:.4f}".ljust(62) + "║")
    print("║" + f"    Recall@5:    {best_ret[1]['recall'][5]:.4f}".ljust(62) + "║")
    print("╠" + "═"*62 + "╣")
    print("║" + "  HALLUCINATION DETECTION".ljust(62) + "║")
    print("║" + f"    Optimal threshold:     {OPTIMAL_THRESHOLD:.2f}".ljust(62) + "║")
    print("║" + f"    NLI F1 (test):         {nli_f1_v5:.4f}".ljust(62) + "║")
    print("║" + f"    Pipeline F1 (test):    {pipe_f1_v5:.4f}".ljust(62) + "║")
    print("║" + f"    NLI F1 (mean±std):     {np.mean(nli_f1s):.4f} ± {np.std(nli_f1s):.4f}".ljust(62) + "║")
    print("║" + f"    Pipeline F1 (mean±std): {np.mean(pipe_f1s):.4f} ± {np.std(pipe_f1s):.4f}".ljust(62) + "║")
    print("╠" + "═"*62 + "╣")
    print("║" + "  CALIBRATION".ljust(62) + "║")
    print("║" + f"    ECE: {calibration['ece']:.4f}".ljust(62) + "║")
    print("╠" + "═"*62 + "╣")
    print("║" + "  BASELINES (best F1)".ljust(62) + "║")
    for name, f1 in sorted(baseline_results.items(), key=lambda x: -x[1])[:5]:
        marker = " <<<" if name == 'Full Pipeline' else ""
        print("║" + f"    {name}: {f1:.4f}{marker}".ljust(62) + "║")
    print("╠" + "═"*62 + "╣")
    print("║" + "  ERROR TAXONOMY".ljust(62) + "║")
    total_errs = sum(len(v) for v in error_tax.values())
    for etype, errs in sorted(error_tax.items(), key=lambda x: -len(x[1])):
        if errs:
            pct = len(errs) / total_errs * 100 if total_errs > 0 else 0
            print("║" + f"    {etype}: {len(errs)} ({pct:.0f}%)".ljust(62) + "║")
    print("╚" + "═"*62 + "╝")
    print()
    print("GitHub: https://github.com/Devarsh-04/halluci-guard-nlp")

print_v5_dashboard()


## 12. Discussion

The results should be interpreted around three main findings.

First, retrieval quality is central. A verifier can only assess groundedness against evidence it actually retrieves, so improvements from cross-encoder or hybrid retrieval directly support safer answer verification.

Second, semantic verification is more appropriate than shallow text matching. NLI is expected to be the strongest single signal because many hallucinations are semantically contradictory while remaining lexically similar to the source.

Third, hard negatives make the evaluation more realistic. Subtle numerical changes, wrong visa subclasses, or unsupported policy details are closer to practical hallucination risks than obvious fabricated statements.

The project therefore contributes a careful verification workflow rather than a claim of production-ready visa advice.


## 13. Limitations

This project has important limitations.

- **Dataset scale:** The dataset is curated and modest in size. It is suitable for coursework and controlled analysis, but it is not a large-scale benchmark.
- **Generator quality:** Flan-T5-base is lightweight and may produce terse or incomplete answers. The main contribution is verification, not high-quality answer generation.
- **Domain scope:** The system is tested on Australian visa information only. Results may not transfer directly to other legal, medical, or financial domains.
- **Synthetic construction:** Some hallucinated examples are manually or synthetically constructed. This supports controlled testing but may not cover the full diversity of real LLM hallucinations.
- **Runtime constraints:** The design favours models that can run in free Colab, which limits the size of the generator and some evaluation choices.

These limitations mean the results should be described as evidence of feasibility, not production deployment readiness.


## 14. Future Work

Future work could strengthen the project in several directions:

1. Evaluate on a larger benchmark of real generated answers from multiple LLMs.
2. Add source attribution at sentence level so each generated claim is linked to evidence.
3. Test stronger instruction-tuned generators while keeping the verification layer model-agnostic.
4. Extend the dataset to more domains and compare domain transfer performance.
5. Improve calibration with temperature scaling or validation-based probability calibration.
6. Use human evaluation to judge whether explanations are understandable and useful.


## 15. Conclusion

HalluciGuard v5 presents a retrieval-augmented NLP workflow for detecting hallucinations in domain-specific QA. The system combines dense and hybrid retrieval, lightweight generation, and a multi-signal verification pipeline based on NER, NLI, and embedding similarity.

The strongest result is not that the system generates perfect answers, but that semantic verification provides a structured way to identify unsupported or contradictory claims. The expanded hard-negative evaluation, ablation studies, calibration analysis, robustness probes, and error taxonomy make the project suitable for a rigorous academic discussion while preserving a clear practical motivation.


## 16. Appendix

The appendix keeps additional demonstrations and legacy summary visualisations. These are useful for presentation rehearsal and report screenshots, but the main narrative should rely on Sections 4-11.


### 16.1 Interactive System Demonstration

The demo prints the retrieved evidence, generated answer, and verification decision for representative user questions.


In [ ]:
def demo(query):
    print(f"\n{'='*60}")
    print(f"Q: {query}")

    info = analyse_query(query)
    print(f"Key terms: {info['nouns']}")

    retrieved = retrieve_passages(query, top_k=5)
    print(f"Retrieved: [{retrieved[0]['passage']['category']}] (ce={retrieved[0]['ce_score']:.3f})")

    answer, _ = generate_answer(query, retrieved)
    print(f"A: {answer}")

    r = detect_hallucination(retrieved[0]['text'], answer)
    emoji = {"GROUNDED":"✅","PARTIALLY GROUNDED":"⚠️","HALLUCINATED":"❌"}
    print(f"{emoji.get(r['verdict'],'')} {r['verdict']} (score={r['combined']:.3f})")
    print(f"   NER={r['ner']:.3f} NLI={r['nli']:.3f} EMB={r['emb']:.3f}")

for q in [
    "Can I work while studying in Australia on a student visa?",
    "How do I become an Australian citizen?",
    "Do I need health insurance as a student?",
    "How much does the partner visa cost?",
    "Can NZ citizens live permanently in Australia?",
]:
    demo(q)

### 16.2 Legacy Comprehensive Visual Summary

This plot is retained from the original notebook because it combines training loss, F1 comparison, confusion matrices, distillation trade-off, and robustness outcomes.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Training loss
axes[0,0].plot(cnn_losses, label='Text CNN', color='#e74c3c', lw=2)
axes[0,0].plot(lstm_losses, label='BiLSTM', color='#3498db', lw=2)
axes[0,0].set_title('Training Loss', fontweight='bold')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# 2. F1 comparison
models_f1 = {
    'Text CNN': f1_score(cnn_true, cnn_pred, average='weighted'),
    'BiLSTM': f1_score(lstm_true, lstm_pred, average='weighted'),
    'NLI': det_results['nli']['f1'],
    'Pipeline': det_results['pipeline']['f1'],
}
colors = ['#e74c3c','#3498db','#2ecc71','#9b59b6']
bars = axes[0,1].bar(models_f1.keys(), models_f1.values(), color=colors)
for b, v in zip(bars, models_f1.values()):
    axes[0,1].text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{v:.3f}',
                   ha='center', fontweight='bold', fontsize=9)
axes[0,1].set_title('F1 Score Comparison', fontweight='bold')
axes[0,1].set_ylim(0, 1.15)

# 3. NLI confusion matrix
cm = confusion_matrix(det_results['true'], det_results['nli']['preds'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,2],
            xticklabels=['Halluc.','Ground.'], yticklabels=['Halluc.','Ground.'])
axes[0,2].set_title('NLI Confusion Matrix', fontweight='bold')

# 4. Pipeline confusion matrix
cm2 = confusion_matrix(det_results['true'], det_results['pipeline']['preds'])
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens', ax=axes[1,0],
            xticklabels=['Halluc.','Ground.'], yticklabels=['Halluc.','Ground.'])
axes[1,0].set_title('Pipeline Confusion Matrix', fontweight='bold')

# 5. Distillation
x = np.arange(2)
axes[1,1].bar(x-0.15, [full_r['f1'], full_r['ms']/100], 0.3, label='Full', color='#2ecc71')
axes[1,1].bar(x+0.15, [dist_r['f1'], dist_r['ms']/100], 0.3, label='Distilled', color='#e67e22')
axes[1,1].set_xticks(x); axes[1,1].set_xticklabels(['F1', 'Latency (x100ms)'])
axes[1,1].set_title('Distillation Trade-off', fontweight='bold'); axes[1,1].legend()

# 6. Robustness
types = ['adversarial','out_of_domain','paraphrased']
for j, v in enumerate(['GROUNDED','PARTIALLY GROUNDED','HALLUCINATED']):
    vals = [sum(1 for r in rob_results if r['type']==t and r['verdict']==v) for t in types]
    axes[1,2].bar(np.arange(3)+j*0.25, vals, 0.25,
                  label=v, color=['#2ecc71','#f39c12','#e74c3c'][j])
axes[1,2].set_xticks(np.arange(3)+0.25)
axes[1,2].set_xticklabels(['Adversarial','OOD','Paraphrased'], fontsize=9)
axes[1,2].set_title('Robustness Results', fontweight='bold'); axes[1,2].legend(fontsize=7)

plt.tight_layout()
plt.savefig('halluciguard_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: halluciguard_results.png")

### 16.3 AI Assistance Acknowledgement Placeholder

If AI tools were used to help draft, revise, debug, or reorganise this notebook/report, acknowledge them in the final written submission according to the assignment instructions. Example wording:

> AI tools were used to assist with code organisation, wording refinement, and notebook structure. The group reviewed, edited, and validated the final content and remains responsible for the submitted work.
